In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes sacrebleu bert-score scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 re

Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 -> Retrieval

In [ ]:
# @title ADD FULL METRICS: Semantic@1 + Recall@3 + MRR@3 + BERTScoreF1

# ============================================================
# Бұл блокты негізгі кодтан КЕЙІН бөлек Colab cell ретінде іске қосыңыз.
#
# Ол қайтадан:
#   Dataset -> FEMSeg -> MAPP -> Qwen
# кезеңдерін жүргізбейді.
#
# Ол дайын pred_answers / topk_answers негізінде:
#   Semantic@1
#   Recall@3
#   MRR@3
#   BERTScoreF1@1
# метрикаларын есептейді.
# ============================================================

!pip install -q bert-score

import time
import json
import numpy as np
import pandas as pd
import torch

from bert_score import score as bert_score


# ============================================================
# 1) CONFIG FOR FULL METRICS
# ============================================================

RUN_ANSWER_SEMANTIC = True
RUN_BERTSCORE = True

# BERTScore үшін жылдамырақ нұсқа.
# Егер мақалаға сапалырақ, бірақ ауыр нұсқа керек болса:
# BERTSCORE_MODEL_TYPE = "xlm-roberta-large"
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"

BERTSCORE_BATCH_SIZE = 16

FULL_METRICS_CSV_PATH = "fast_qwen25_femseg_mapp_FULL_METRICS.csv"


# ============================================================
# 2) CHECK REQUIRED OBJECTS
# ============================================================

required_names = [
    "_eval_cache",
    "precompute_eval_predictions",
    "_prepare_answer_semantic_cache",
    "evaluate_exact_at_1",
    "evaluate_top3_exact_hit",
    "evaluate_token_f1_at_1",
    "evaluate_mean_cos_qsim_at_1",
    "evaluate_semantic_at_1_ans_cos",
    "evaluate_recall_at_3_ans_cos",
    "evaluate_mrr_at_3_ans_cos",
    "norm_for_exact",
    "token_f1",
    "stable_item_key",
    "SEM_THR",
    "TOPK_EVAL",
    "train_answers",
    "test_answers",
]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        "Мына объектілер табылмады. Алдымен негізгі Qwen2.5 кодын толық іске қосыңыз:\n"
        + "\n".join(missing)
    )


# ============================================================
# 3) BERTScore FUNCTION
# ============================================================

def compute_bertscore_f1_at_1():
    if "bertscore_itemwise" in _eval_cache and _eval_cache["bertscore_itemwise"] is not None:
        itemwise = _eval_cache["bertscore_itemwise"]
        return float(np.mean(itemwise)), itemwise

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\nBERTScoreF1@1 есептелуде...")
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)

    t0 = time.time()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        _, _, f1 = bert_score(
            preds,
            trues,
            model_type=BERTSCORE_MODEL_TYPE,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    except Exception as e:
        print("Бірінші BERTScore есептеу сәтсіз болды:", repr(e))
        print("Fallback: lang='kk' арқылы есептеу басталды...")

        _, _, f1 = bert_score(
            preds,
            trues,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    if hasattr(f1, "detach"):
        itemwise = f1.detach().cpu().numpy().astype(float)
    else:
        itemwise = np.array(f1).astype(float)

    _eval_cache["bertscore_itemwise"] = itemwise
    _eval_cache["bertscore_f1_mean"] = float(np.mean(itemwise))
    _eval_cache["bertscore_model_type"] = BERTSCORE_MODEL_TYPE

    print(f"BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")

    return float(np.mean(itemwise)), itemwise


# ============================================================
# 4) DETAILS CSV WITH ALL METRICS
# ============================================================

def build_full_metrics_details():
    precompute_eval_predictions()
    _prepare_answer_semantic_cache()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    questions = _eval_cache["test_questions"]

    qsims = _eval_cache["pred_q_sims"]
    msims = _eval_cache["pred_morph_sims"]
    rscores = _eval_cache["pred_rerank_scores"]

    topk_indices = _eval_cache["topk_indices"]
    topk_qsims = _eval_cache["topk_qsims"]
    topk_answers = _eval_cache["topk_answers"]

    ans_cos = _eval_cache["ans_cos"]
    topk_ans_cos = _eval_cache["topk_ans_cos"]

    if RUN_BERTSCORE:
        bert_mean, bert_itemwise = compute_bertscore_f1_at_1()
    else:
        bert_mean, bert_itemwise = None, None

    rows = []

    for i, (q, gold, pred) in enumerate(zip(questions, trues, preds)):
        top_answers = topk_answers[i]

        top_exact_flags = [
            float(norm_for_exact(a) == norm_for_exact(gold))
            for a in top_answers
        ]

        top_sem_flags = [
            float(x >= SEM_THR)
            for x in topk_ans_cos[i].tolist()
        ]

        recall3 = float(any(top_sem_flags))

        mrr3 = 0.0
        for rank, flag in enumerate(top_sem_flags, start=1):
            if flag == 1.0:
                mrr3 = 1.0 / rank
                break

        row = {
            "item_key": stable_item_key(q, gold),
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,

            "QSim": float(qsims[i]),
            "MorphSim": float(msims[i]),
            "RerankScore": float(rscores[i]),

            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "Top3ExactHit": float(any(top_exact_flags)),
            "TokenF1": float(token_f1(pred, gold)),

            "AnsCos": float(ans_cos[i]),
            "SemHit": float(ans_cos[i] >= SEM_THR),
            "Recall@3": recall3,
            "MRR@3": float(mrr3),

            "Top3_indices": json.dumps(topk_indices[i].tolist(), ensure_ascii=False),
            "Top3_QSims": json.dumps(
                [round(float(x), 6) for x in topk_qsims[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_pred_answers": json.dumps(top_answers, ensure_ascii=False),
            "Top3_AnsCos": json.dumps(
                [round(float(x), 6) for x in topk_ans_cos[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_ExactFlags": json.dumps(top_exact_flags, ensure_ascii=False),
            "Top3_SemFlags": json.dumps(top_sem_flags, ensure_ascii=False),
        }

        if bert_itemwise is not None:
            row["BERTScoreF1"] = float(bert_itemwise[i])

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 5) FINAL FULL METRICS EVALUATION
# ============================================================

def evaluate_full_metrics():
    t0 = time.time()

    precompute_eval_predictions()

    print("\nAnswer semantic metrics есептелуде...")
    _prepare_answer_semantic_cache()

    exact, exact_vals = evaluate_exact_at_1()
    top3_exact, top3_exact_vals = evaluate_top3_exact_hit()
    token_f1_mean, token_f1_vals = evaluate_token_f1_at_1()
    qsim_mean, qsim_vals = evaluate_mean_cos_qsim_at_1()

    sem1, sem1_vals = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    recall3, recall3_vals = evaluate_recall_at_3_ans_cos(threshold=SEM_THR)
    mrr3, mrr3_vals = evaluate_mrr_at_3_ans_cos(threshold=SEM_THR)

    if RUN_BERTSCORE:
        bertscore_f1, bertscore_vals = compute_bertscore_f1_at_1()
    else:
        bertscore_f1, bertscore_vals = None, None

    details_df = build_full_metrics_details()
    details_df.to_csv(FULL_METRICS_CSV_PATH, index=False, encoding="utf-8-sig")

    print("\n==================== FULL METRICS RESULTS ====================")
    print("Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 -> Retrieval")
    print(f"QWEN_MODEL_NAME              : {QWEN_MODEL_NAME}")
    print(f"Dataset rows used            : {len(qa_data)}")
    print(f"Test rows                    : {len(test_answers)}")
    print(f"SEM_THR                      : {SEM_THR}")
    print(f"TOPK_EVAL                    : {TOPK_EVAL}")

    print(f"Exact@1                      : {exact:.6f}")
    print(f"Top-3 ExactHit               : {top3_exact:.6f}")
    print(f"TokenF1@1                    : {token_f1_mean:.6f}")
    print(f"MeanCos@1(QSim)              : {qsim_mean:.6f}")

    print(f"Semantic@1(ans_cos>={SEM_THR}) : {sem1:.6f}")
    print(f"Recall@3(ans_cos>={SEM_THR})   : {recall3:.6f}")
    print(f"MRR@3(ans_cos>={SEM_THR})      : {mrr3:.6f}")

    if bertscore_f1 is not None:
        print(f"BERTScoreF1@1                : {bertscore_f1:.6f}")
        print(f"BERTScore model              : {_eval_cache.get('bertscore_model_type')}")
    else:
        print("BERTScoreF1@1                : N/A")

    print(f"Evaluation time              : {time.time() - t0:.2f} sec")
    print(f"CSV saved                    : {FULL_METRICS_CSV_PATH}")

    return details_df


# ============================================================
# 6) RUN
# ============================================================

full_details_df = evaluate_full_metrics()
display(full_details_df.head(5))


Answer semantic metrics есептелуде...

Answer semantic embeddings есептелуде...
Encoded 595/595 | 6.1 сек
Encoded 595/595 | 5.8 сек
Encoded 1000/1785 | 10.1 сек
Encoded 1785/1785 | 17.8 сек
Answer semantic embeddings дайын: 29.78 сек

BERTScoreF1@1 есептелуде...
BERTScore model: bert-base-multilingual-cased


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/68 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/38 [00:00<?, ?it/s]

done in 3.16 seconds, 188.37 sentences/sec
BERTScoreF1@1 дайын: 16.35 сек

==================== FULL METRICS RESULTS ====================
Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 -> Retrieval
QWEN_MODEL_NAME              : Qwen/Qwen2.5-1.5B-Instruct
Dataset rows used            : 5944
Test rows                    : 595
SEM_THR                      : 0.85
TOPK_EVAL                    : 3
Exact@1                      : 0.085714
Top-3 ExactHit               : 0.099160
TokenF1@1                    : 0.402280
MeanCos@1(QSim)              : 0.986904
Semantic@1(ans_cos>=0.85) : 0.774790
Recall@3(ans_cos>=0.85)   : 0.857143
MRR@3(ans_cos>=0.85)      : 0.812045
BERTScoreF1@1                : 0.801605
BERTScore model              : bert-base-multilingual-cased
Evaluation time              : 46.43 sec
CSV saved                    : fast_qwen25_femseg_mapp_FULL_METRICS.csv


,item_key,test_question,gold_answer,pred_answer,QSim,MorphSim,RerankScore,Exact,Top3ExactHit,TokenF1,...,SemHit,Recall@3,MRR@3,Top3_indices,Top3_QSims,Top3_pred_answers,Top3_AnsCos,Top3_ExactFlags,Top3_SemFlags,BERTScoreF1
0,d80b7ef941e26fccdbfee4bd547757c0,Сөздіктегі кілт-мағына жұбын қалай жоямыз?,pop() әдісін қолданамыз. Мысалы: sozdik = {'a'...,clear() әдісін қолданамыз. Мысалы: sozdik = {'...,0.985981,0.402101,0.845268,0.0,0.0,0.774194,...,1.0,1.0,1.0,"[1671, 1606, 4361]","[0.985981, 0.991675, 0.990523]","[""clear() әдісін қолданамыз. Мысалы: sozdik = ...","[0.96129, 0.130855, 0.167132]","[0.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.943826
1,ba8c6e07e7c917eb1c47395ecc03d7b4,GitHub Actions-те орналастыруды қалай автоматт...,"AWS, Azure немесе SSH арқылы серверге кодты жі...",Тапсырмаларды конфигурация файлына сүйеніп іск...,0.986910,0.360714,0.835270,0.0,0.0,0.000000,...,1.0,1.0,1.0,"[3043, 4903, 3154]","[0.98691, 0.985496, 0.985177]","[""Тапсырмаларды конфигурация файлына сүйеніп і...","[0.961142, 0.959456, 0.951013]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.714074
2,0ab1409f6d7c527b4d8efb365eca6ff9,Боттың қолжетімділігін қалай мониторинг жасаймыз?,UptimeRobot немесе custom logging арқылы.,pytest-monitor немесе custom logging арқылы.,0.988082,0.465238,0.861840,0.0,0.0,0.727273,...,1.0,1.0,1.0,"[2004, 1634, 913]","[0.988082, 0.989383, 0.988337]","[""pytest-monitor немесе custom logging арқылы....","[0.968742, 0.949889, 0.953776]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.860071
3,8cad88a5baf2b3ddedb2fef91a11ca4c,Функцияда қалай прогресс барын көрсетеміз?,tqdm модулін қолданамыз. Мысалы: from tqdm imp...,pydub модулін қолданамыз. Мысалы: from pydub i...,0.928307,0.360714,0.813294,0.0,0.0,0.625000,...,1.0,1.0,1.0,"[3726, 5138, 2155]","[0.928307, 0.935621, 0.982198]","[""pydub модулін қолданамыз. Мысалы: from pydub...","[0.881126, 0.823463, 0.806339]","[0.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.841337
4,8509cae4e0587b89ab3eeb87fec5b8d0,GET пен POST сұрауларының басты айырмашылығы н...,"GET деректерді алады, POST деректерді жібереді...","Rebase тарихты сызықты етеді, merge тармақтард...",0.956154,0.391667,0.831474,0.0,0.0,0.000000,...,1.0,1.0,1.0,"[2307, 1989, 2129]","[0.956154, 0.963591, 0.976421]","[""Rebase тарихты сызықты етеді, merge тармақта...","[0.940062, 0.909851, 0.930405]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.704101


Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 -> Retrieval->Fine-tuning

In [ ]:
# @title FEMSeg + MAPP -> Qwen2.5 Fine-tuning -> Retrieval | Full Metrics | Google Colab Ready

# ============================================================
# GOOGLE COLAB READY CODE
#
# ҚАТАҢ САҚТАЛҒАН ҚҰРЫЛЫМ:
#
# Dataset
#   -> clean text
#   -> FEMSeg segmentation
#   -> MAPP normalization
#   -> q_dense = clean + FEMSeg + MAPP
#   -> Qwen2.5 fine-tuning
#   -> Retrieval
#   -> Full Metrics
#
# NO GENERATION
# ENCODER-STYLE RETRIEVAL
# LoRA / QLoRA contrastive fine-tuning
#
# Metrics:
#   Exact@1
#   Top-3 ExactHit
#   TokenF1@1
#   MeanCos@1(QSim)
#   Semantic@1(ans_cos >= 0.85)
#   Recall@3(ans_cos >= 0.85)
#   MRR@3(ans_cos >= 0.85)
#   BERTScoreF1@1
# ============================================================


# ============================================================
# 0) INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers accelerate bitsandbytes peft bert-score pytorch-crf scikit-learn


# ============================================================
# 1) IMPORTS AND ENVIRONMENT
# ============================================================

import os
os.environ["WANDB_DISABLED"] = "true"

import re
import glob
import time
import json
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Set
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

from bert_score import score as bert_score

try:
    from torchcrf import CRF
except Exception:
    CRF = None

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = print


if CRF is None:
    raise ImportError(
        "torchcrf табылмады. Colab-та Runtime -> Restart runtime жасап, кодты қайта іске қосыңыз."
    )


# ============================================================
# 2) SPEED / SEED SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE != "cuda":
    raise RuntimeError(
        "Qwen2.5 fine-tuning үшін GPU керек. Colab-та Runtime -> Change runtime type -> GPU таңдаңыз."
    )


# ============================================================
# 3) CONFIG
# ============================================================

DATA_JSON_PATH = "grok_python_dataset.json"

TEST_SIZE = 0.10
SEM_THR = 0.85
TOPK_EVAL = 3

# 5000 QA көлеміне лайықты fine-tuning параметрлері
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

USE_4BIT = True

NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
TEMPERATURE = 0.05
MAX_GRAD_NORM = 1.0

MAX_Q_LEN = 160
MAX_A_LEN = 224

EVAL_ENCODE_BATCH_SIZE = 8
SIM_BATCH_SIZE = 256

QWEN_MAX_LENGTH_FOR_RETRIEVAL = 192

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# FEMSeg paths
FEMSEG_CHAR2ID = "/content/drive/MyDrive/KAZ_MORPH/char2id_femseg_v3_50k.json"
FEMSEG_CKPT = "/content/drive/MyDrive/KAZ_MORPH/femseg_v3_50k_epoch3.pt"

# MAPP config
USE_MAPP = True
MAPP_TAG = "[MAPP]"
FEMSEG_TAG = "[FEMSEG]"
MAPP_MIN_WORD_LEN = 4

# Hybrid retrieval/reranking
USE_MORPH_RERANK = True
DENSE_TOPK = 30
HYBRID_DENSE_WEIGHT = 0.75
HYBRID_MORPH_WEIGHT = 0.25

# Full metrics
RUN_BERTSCORE = True
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
# Егер сапалырақ, бірақ ауыр нұсқа керек болса:
# BERTSCORE_MODEL_TYPE = "xlm-roberta-large"

BERTSCORE_BATCH_SIZE = 16

RUN_BOOTSTRAP_CI = False
BOOT_N = 500

EXPORT_CSV = True
CSV_PATH = "femseg_mapp_qwen25_finetuned_full_metrics.csv"

SAVE_LORA = True
LORA_SAVE_DIR = "/content/drive/MyDrive/KAZ_MORPH/qwen25_15b_lora_femseg_mapp_qa" if IN_COLAB else "./qwen25_15b_lora_femseg_mapp_qa"


# ============================================================
# 4) GOOGLE DRIVE
# ============================================================

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)


# ============================================================
# 5) ROBUST JSON LOADING
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p

    candidates = [
        f"/content/{p}",
        f"/content/drive/MyDrive/{p}",
        f"/content/drive/MyDrive/KAZ_MORPH/{p}",
    ]

    for c in candidates:
        if Path(c).exists():
            return c

    name = Path(p).name
    hits = glob.glob(f"**/{name}", recursive=True)

    if hits:
        return hits[0]

    near = glob.glob("**/*.json", recursive=True)

    raise FileNotFoundError(
        f"Файл табылмады: {p}\n"
        f"Табылған .json файлдар:\n" + "\n".join(near[:30])
    )


def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)


def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)


def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text


def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None

    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    return None


def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)

    if normalized is not None:
        return [normalized]

    if isinstance(obj, list):
        out = []

        for item in obj:
            norm = _normalize_record(item)

            if norm is not None:
                out.append(norm)

        return out

    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                out = []

                for item in obj[key]:
                    norm = _normalize_record(item)

                    if norm is not None:
                        out.append(norm)

                return out

    return []


def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    obj = json.loads(_normalize_json_text(text))
    records = _extract_records(obj)

    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")

    return records


def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx = 0
    records = []

    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1

        if idx >= len(text):
            break

        obj, end = decoder.raw_decode(text, idx)
        idx = end

        extracted = _extract_records(obj)

        if extracted:
            records.extend(extracted)

    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")

    return records


def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records = []
    bad_lines = []

    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()

        if not s:
            continue

        if s.endswith(","):
            s = s[:-1].strip()

        s = _normalize_json_text(s)

        try:
            obj = json.loads(s)
            extracted = _extract_records(obj)

            if extracted:
                records.extend(extracted)
            else:
                bad_lines.append((line_no, "QA өрістері табылмады", s[:200]))

        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))

    if not records:
        preview = "\n".join(
            [f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]]
        )

        raise ValueError(
            "Файлдан бірде-бір QA жазба оқылмады.\n"
            f"Алғашқы қате жолдар:\n{preview}"
        )

    if bad_lines:
        print(f"Ескерту: {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")

    return records


def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []

    for parser in (
        _parse_as_single_json,
        _parse_as_multiple_json_objects,
        _parse_line_by_line,
    ):
        try:
            records = parser(text)

            if records:
                return records

        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")

    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


data_path = find_data_path(DATA_JSON_PATH)
print("QA JSON файл:", data_path)

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON ішінде question/answer немесе instruction/response болуы керек."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"] = qa_data["answer"].astype(str).str.strip()

qa_data = qa_data[
    (qa_data["question"] != "") &
    (qa_data["answer"] != "")
].reset_index(drop=True)

print("Барлық QA саны:", len(qa_data))
display(qa_data.head(3))


# ============================================================
# 6) BASIC TEXT PROCESSING AND METRICS
# ============================================================

_punct_space_left = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space = re.compile(r"\s+")


def clean_text(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_text(text).lower()).strip()


def tokens(text: str) -> List[str]:
    t = clean_text(text).lower()

    return re.findall(
        r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+",
        t
    )


def token_f1(pred: str, gold: str) -> float:
    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    pc = Counter(p)
    gc = Counter(g)

    inter = sum((pc & gc).values())

    if inter == 0:
        return 0.0

    precision = inter / max(1, len(p))
    recall = inter / max(1, len(g))

    return (2 * precision * recall) / (precision + recall + 1e-12)


def stable_item_key(test_question: str, gold_answer: str) -> str:
    raw = norm_for_exact(test_question) + " ||| " + norm_for_exact(gold_answer)
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


# ============================================================
# 7) FEMSeg MODEL
# ============================================================

BMES_TAGS = ["B", "M", "E", "S"]
TAG2ID = {t: i for i, t in enumerate(BMES_TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}


class FEMSegV3(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int = 128,
        lstm_hidden_dim: int = 256,
        dropout: float = 0.3,
        num_tags: int = len(BMES_TAGS),
    ):
        super().__init__()

        self.emb = nn.Embedding(
            vocab_size,
            char_emb_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=char_emb_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.fc = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward_features(self, x, mask):
        emb = self.emb(x)
        emb = self.dropout(emb)
        h, _ = self.lstm(emb)
        h = self.dropout(h)
        return h

    def forward_logits(self, x, mask):
        feats = self.forward_features(x, mask)
        return self.fc(feats)

    def forward(self, x, tags=None, mask=None):
        if mask is None:
            mask = x != 0

        emissions = self.forward_logits(x, mask)

        if tags is not None:
            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(emissions, mask=mask)


def bmes_to_cse(chars: List[str], tags: List[str]) -> str:
    morphs = []
    cur = ""

    for ch, t in zip(chars, tags):
        if t == "B":
            if cur:
                morphs.append(cur)
            cur = ch

        elif t == "M":
            cur += ch

        elif t == "E":
            cur += ch
            morphs.append(cur)
            cur = ""

        elif t == "S":
            if cur:
                morphs.append(cur)
            morphs.append(ch)
            cur = ""

    if cur:
        morphs.append(cur)

    return "@@ ".join(morphs)


class KazMorphSegmentor:
    def __init__(self, char2id_path: str, ckpt_path: str, use_cuda: bool = True):
        with open(char2id_path, "r", encoding="utf-8") as f:
            self.char2id = json.load(f)

        self.pad_id = 0
        self.unk_id = self.char2id.get("<unk>") or self.char2id.get("[UNK]") or 1

        self.device = torch.device(
            "cuda" if (use_cuda and torch.cuda.is_available()) else "cpu"
        )

        vocab_size = max(self.char2id.values()) + 1

        self.model = FEMSegV3(vocab_size=vocab_size)
        self.model.to(self.device)

        state = torch.load(ckpt_path, map_location=self.device)

        if isinstance(state, dict):
            if "state_dict" in state:
                state = state["state_dict"]
            elif "model_state_dict" in state:
                state = state["model_state_dict"]

        if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
            state = {
                k.replace("module.", "", 1): v
                for k, v in state.items()
            }

        self.model.load_state_dict(state, strict=True)
        self.model.eval()

        self.word_cache = {}
        self.text_cache = {}

    def _word_to_tensor(self, word: str):
        ids = [
            self.char2id.get(ch, self.unk_id)
            for ch in list(word)
        ]

        x = torch.tensor(
            [ids],
            dtype=torch.long,
            device=self.device
        )

        mask = x != self.pad_id

        return x, mask

    def segment_word(self, word: str) -> str:
        word = word.strip()

        if not word:
            return ""

        cached = self.word_cache.get(word)

        if cached is not None:
            return cached

        x, mask = self._word_to_tensor(word)

        with torch.no_grad():
            best_paths = self.model(x, mask=mask)

        tags = [ID2TAG[i] for i in best_paths[0]]
        result = bmes_to_cse(list(word), tags)

        self.word_cache[word] = result

        return result

    def segment_text(self, text: str) -> str:
        text = text.strip()

        if not text:
            return ""

        cached = self.text_cache.get(text)

        if cached is not None:
            return cached

        words = text.split()
        out_words = [self.segment_word(w) for w in words]

        result = " | ".join(out_words)
        self.text_cache[text] = result

        return result


assert os.path.exists(FEMSEG_CHAR2ID), f"FEMSeg char2id табылмады: {FEMSEG_CHAR2ID}"
assert os.path.exists(FEMSEG_CKPT), f"FEMSeg checkpoint табылмады: {FEMSEG_CKPT}"

print("FEMSeg жүктелуде...")

femseg = KazMorphSegmentor(
    char2id_path=FEMSEG_CHAR2ID,
    ckpt_path=FEMSEG_CKPT,
    use_cuda=True,
)

print("FEMSeg дайын.")


def femseg_to_sp(text: str) -> str:
    text = clean_text(str(text))

    if not text:
        return ""

    seg_line = femseg.segment_text(text)
    sp_tokens = []

    for word_seg in seg_line.split(" | "):
        word_seg = word_seg.strip()

        if not word_seg:
            continue

        morphs = [
            m.strip()
            for m in word_seg.split("@@ ")
            if m.strip()
        ]

        if not morphs:
            continue

        sp_tokens.append("▁" + morphs[0])
        sp_tokens.extend(morphs[1:])

    return " ".join(sp_tokens)


def sp_token_set(sp_text: str) -> Set[str]:
    out = set()

    for tok in str(sp_text).split():
        tok = tok.replace("▁", "").strip()

        if tok:
            out.add(tok)

    return out


print("FEMSeg тест:", femseg_to_sp("Мен мектепке барамын"))


# ============================================================
# 8) MAPP NORMALIZATION
# ============================================================

WORD_RE = re.compile(
    r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9_+#-]+",
    re.UNICODE
)

PROTECTED_WORDS = {
    "не", "неге", "неліктен", "қалай", "қайда", "қайдан", "қашан",
    "қандай", "қанша", "қай", "кім", "кімге", "кімнің", "неше",

    "python", "def", "for", "while", "if", "elif", "else", "print", "input",
    "list", "dict", "tuple", "set", "int", "str", "float", "bool", "none",
    "true", "false", "class", "return", "break", "continue", "range", "len",
    "append", "insert", "remove", "pop", "sort", "lambda", "map", "filter",
    "reduce", "zip", "enumerate", "and", "or", "not", "in", "is",
}

MAPP_WORD_MAP = {
    "калай": "қалай",
    "кандай": "қандай",
    "кайда": "қайда",
    "кашан": "қашан",
    "канша": "қанша",
    "аныкта": "анықта",
    "аныктайды": "анықтайды",
    "аныктаймыз": "анықтаймыз",
    "есептеймз": "есептейміз",
    "пайтон": "python",
    "питон": "python",
    "функсия": "функция",
    "функсиа": "функция",
    "циклд": "цикл",
    "стр": "str",
    "инт": "int",
    "бул": "bool",
    "принт": "print",
    "инпут": "input",
}

_mapp_word_cache = {}
_mapp_text_cache = {}


def _basic_mapp_word_norm(word: str) -> str:
    w = str(word).strip().lower()
    return MAPP_WORD_MAP.get(w, w)


def _is_ascii_programming_token(word: str) -> bool:
    return bool(re.fullmatch(r"[a-z0-9_+#-]+", word))


def _split_seg_word(seg_word: str) -> List[str]:
    return [
        m.strip()
        for m in str(seg_word).split("@@ ")
        if m.strip()
    ]


def _mapp_stem_from_femseg(word: str) -> str:
    w = _basic_mapp_word_norm(word)

    if not USE_MAPP:
        return w

    if not w:
        return w

    cached = _mapp_word_cache.get(w)

    if cached is not None:
        return cached

    if w in PROTECTED_WORDS:
        _mapp_word_cache[w] = w
        return w

    if len(w) < MAPP_MIN_WORD_LEN:
        _mapp_word_cache[w] = w
        return w

    if w.isdigit():
        _mapp_word_cache[w] = w
        return w

    if _is_ascii_programming_token(w):
        _mapp_word_cache[w] = w
        return w

    try:
        seg_word = femseg.segment_word(w)
        morphs = _split_seg_word(seg_word)

        if not morphs:
            result = w
        else:
            stem = _basic_mapp_word_norm(morphs[0])
            result = stem if len(stem) >= 2 else w

    except Exception:
        result = w

    _mapp_word_cache[w] = result

    return result


def mapp_normalize(text: str) -> str:
    t = clean_text(text).lower().strip()

    if not t:
        return ""

    cached = _mapp_text_cache.get(t)

    if cached is not None:
        return cached

    words = WORD_RE.findall(t)
    out = []
    prev = None

    for w in words:
        nw = _mapp_stem_from_femseg(w)

        if nw and nw != prev:
            out.append(nw)
            prev = nw

    result = " ".join(out).strip()

    if not result:
        result = t

    _mapp_text_cache[t] = result

    return result


def compose_dense_view(q_clean: str, q_sp: str, q_mapp: str) -> str:
    q_clean = clean_text(q_clean)
    q_sp = clean_text(q_sp)
    q_mapp = clean_text(q_mapp)

    parts = []

    if q_clean:
        parts.append(q_clean)

    if q_sp:
        parts.append(FEMSEG_TAG)
        parts.append(q_sp)

    if q_mapp and norm_for_exact(q_mapp) != norm_for_exact(q_clean):
        parts.append(MAPP_TAG)
        parts.append(q_mapp)

    return " ".join(parts).strip()


# ============================================================
# 9) ANSWER POST-PROCESSING
# ============================================================

POST_WORD_MAP = {
    "пайтон": "Python",
    "питон": "Python",
    "функсиа": "функция",
    "функсия": "функция",
    "принт": "print",
    "инпут": "input",
}


def _is_code_like_line(line: str) -> bool:
    s = line.rstrip()

    if not s.strip():
        return False

    if s.startswith("    ") or s.startswith("\t"):
        return True

    code_heads = (
        "def ", "for ", "while ", "if ", "elif ", "else:",
        "return ", "class ", "print("
    )

    if s.lstrip().startswith(code_heads):
        return True

    if re.match(r"^\s*#.*$", s):
        return True

    return False


def _replace_whole_words(text: str, mapping: Dict[str, str]) -> str:
    out = text

    for a, b in mapping.items():
        out = re.sub(
            rf"\b{re.escape(a)}\b",
            b,
            out,
            flags=re.IGNORECASE
        )

    return out


def answer_postprocess(text: str) -> str:
    if text is None:
        return ""

    t = str(text).replace("\r\n", "\n").strip()

    if not t:
        return ""

    lines = t.split("\n")
    out_lines = []
    prev_norm = None

    for line in lines:
        raw = line.rstrip()

        if not raw.strip():
            if out_lines and out_lines[-1] != "":
                out_lines.append("")
            prev_norm = None
            continue

        if _is_code_like_line(raw):
            cleaned = raw.rstrip()
        else:
            cleaned = clean_text(raw)
            cleaned = _replace_whole_words(cleaned, POST_WORD_MAP)

        normed = cleaned.strip()

        if normed != prev_norm:
            out_lines.append(cleaned)
            prev_norm = normed

    result = "\n".join(out_lines).strip()
    result = re.sub(r"\n{3,}", "\n\n", result).strip()

    return result


# ============================================================
# 10) BUILD VIEWS BEFORE Qwen2.5
# ============================================================

print("\n1-қадам: clean + FEMSeg + MAPP дайындалуда...")

qa_data["q_clean"] = qa_data["question"].map(clean_text)
qa_data["a_clean"] = qa_data["answer"].map(clean_text)

unique_q_clean = qa_data["q_clean"].unique().tolist()

print("\nFEMSeg segmentation басталды...")
t0 = time.time()

q_sp_map = {}

for i, txt in enumerate(unique_q_clean, start=1):
    q_sp_map[txt] = femseg_to_sp(txt)

    if i % 1000 == 0 or i == len(unique_q_clean):
        print(f"FEMSeg {i}/{len(unique_q_clean)} дайын | {time.time() - t0:.1f} сек")

qa_data["q_sp"] = qa_data["q_clean"].map(q_sp_map)

print("\nMAPP normalization басталды...")
t0 = time.time()

q_mapp_map = {}

for i, txt in enumerate(unique_q_clean, start=1):
    q_mapp_map[txt] = mapp_normalize(txt)

    if i % 1000 == 0 or i == len(unique_q_clean):
        print(f"MAPP {i}/{len(unique_q_clean)} дайын | {time.time() - t0:.1f} сек")

qa_data["q_mapp"] = qa_data["q_clean"].map(q_mapp_map)

qa_data["q_dense"] = qa_data.apply(
    lambda r: compose_dense_view(
        q_clean=r["q_clean"],
        q_sp=r["q_sp"],
        q_mapp=r["q_mapp"]
    ),
    axis=1
)

qa_data["a_final"] = qa_data["a_clean"].map(answer_postprocess)

qa_data["q_tok_set"] = qa_data["q_mapp"].map(lambda x: set(WORD_RE.findall(str(x))))
qa_data["q_seg_set"] = qa_data["q_sp"].map(sp_token_set)

print("\nДайын preprocessing мысалы:")
display(
    qa_data[
        [
            "question",
            "q_clean",
            "q_sp",
            "q_mapp",
            "q_dense",
            "a_final",
        ]
    ].head(3)
)


# ============================================================
# 11) TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    qa_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\nTRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED}")


# ============================================================
# 12) LOAD Qwen2.5 MODEL AFTER FEMSeg + MAPP
# ============================================================

print("\n2-қадам: Qwen2.5 model жүктеледі.")
print("Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 fine-tuning -> Retrieval")
print("QWEN_MODEL_NAME:", QWEN_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

else:
    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(DEVICE)

qwen_model.config.use_cache = False

if USE_4BIT:
    qwen_model = prepare_model_for_kbit_training(
        qwen_model,
        use_gradient_checkpointing=True
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)

qwen_model = get_peft_model(qwen_model, lora_config)

try:
    qwen_model.print_trainable_parameters()
except Exception:
    pass

qwen_model.train()

QWEN_DEVICE = next(qwen_model.parameters()).device
print("Qwen2.5 + LoRA дайын.")
print("QWEN_DEVICE:", QWEN_DEVICE)


# ============================================================
# 13) EMBEDDING FUNCTIONS
# ============================================================

def last_token_pool(
    last_hidden_state: torch.Tensor,
    attention_mask: torch.Tensor
) -> torch.Tensor:
    lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(
        last_hidden_state.size(0),
        device=last_hidden_state.device
    )
    pooled = last_hidden_state[batch_idx, lengths]
    return pooled


def encode_from_batch(batch_enc: Dict[str, torch.Tensor]) -> torch.Tensor:
    out = qwen_model(
        **batch_enc,
        output_hidden_states=False,
        return_dict=True
    )

    pooled = last_token_pool(
        out.last_hidden_state,
        batch_enc["attention_mask"]
    )

    pooled = pooled.float()
    pooled = F.normalize(pooled, p=2, dim=1)

    return pooled


@torch.inference_mode()
def qwen_encode(
    texts: List[str],
    batch_size: int = EVAL_ENCODE_BATCH_SIZE,
    normalize: bool = True,
    max_length: int = QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress: bool = True
) -> np.ndarray:
    qwen_model.eval()

    all_vecs = []
    n = len(texts)
    t0 = time.time()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_texts = [str(x) for x in texts[start:end]]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {
            k: v.to(QWEN_DEVICE)
            for k, v in enc.items()
        }

        out = qwen_model(
            **enc,
            output_hidden_states=False,
            return_dict=True
        )

        pooled = last_token_pool(
            out.last_hidden_state,
            enc["attention_mask"]
        ).float()

        if normalize:
            pooled = F.normalize(pooled, p=2, dim=1)

        all_vecs.append(
            pooled.detach().cpu().numpy()
        )

        if show_progress and ((end % 1000 == 0) or end == n):
            print(f"Encoded {end}/{n} | {time.time() - t0:.1f} сек")

    return np.vstack(all_vecs).astype(np.float32)


# ============================================================
# 14) TRAINING DATASET
# ============================================================

class QAPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.questions = df["q_dense"].tolist()
        self.answers = df["a_final"].tolist()

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return {
            "question": self.questions[idx],
            "answer": self.answers[idx],
        }


def collate_qa_batch(batch):
    questions = [
        item["question"]
        for item in batch
    ]

    answers = [
        item["answer"]
        for item in batch
    ]

    q_enc = tokenizer(
        questions,
        padding=True,
        truncation=True,
        max_length=MAX_Q_LEN,
        return_tensors="pt"
    )

    a_enc = tokenizer(
        answers,
        padding=True,
        truncation=True,
        max_length=MAX_A_LEN,
        return_tensors="pt"
    )

    q_enc = {
        k: v.to(QWEN_DEVICE)
        for k, v in q_enc.items()
    }

    a_enc = {
        k: v.to(QWEN_DEVICE)
        for k, v in a_enc.items()
    }

    return q_enc, a_enc


train_dataset = QAPairDataset(train_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_qa_batch,
    drop_last=True,
)


# ============================================================
# 15) CONTRASTIVE FINE-TUNING
# ============================================================

trainable_params = [
    p for p in qwen_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_update_steps = max(1, (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS)
warmup_steps = int(total_update_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps
)

print("\n==================== TRAINING CONFIG ====================")
print(f"Train size               : {len(train_df)}")
print(f"Test size                : {len(test_df)}")
print(f"Epochs                   : {NUM_EPOCHS}")
print(f"Train batch size          : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation     : {GRAD_ACCUM_STEPS}")
print(f"Effective batch size      : {EFFECTIVE_BATCH_SIZE}")
print(f"Learning rate             : {LEARNING_RATE}")
print(f"Warmup steps              : {warmup_steps}")
print(f"Total update steps        : {total_update_steps}")
print(f"Temperature               : {TEMPERATURE}")
print(f"LoRA r                    : {LORA_R}")
print(f"LoRA alpha                : {LORA_ALPHA}")
print(f"LoRA dropout              : {LORA_DROPOUT}")
print("==========================================================")


def contrastive_loss(
    q_emb: torch.Tensor,
    a_emb: torch.Tensor,
    temperature: float = TEMPERATURE
):
    logits = torch.matmul(q_emb, a_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_q_to_a = F.cross_entropy(logits, labels)
    loss_a_to_q = F.cross_entropy(logits.T, labels)

    return 0.5 * (loss_q_to_a + loss_a_to_q)


qwen_model.train()

global_step = 0
running_loss = 0.0

t_train = time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS} басталды...")
    epoch_t0 = time.time()

    for step, (q_enc, a_enc) in enumerate(train_loader, start=1):
        with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
            q_emb = encode_from_batch(q_enc)
            a_emb = encode_from_batch(a_enc)

            loss = contrastive_loss(q_emb, a_emb)
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if global_step % 25 == 0:
                avg_loss = running_loss / 25
                running_loss = 0.0
                lr_now = scheduler.get_last_lr()[0]
                print(
                    f"epoch={epoch} | update_step={global_step}/{total_update_steps} | "
                    f"loss={avg_loss:.4f} | lr={lr_now:.2e}"
                )

    print(f"Epoch {epoch} аяқталды | уақыт: {time.time() - epoch_t0:.1f} сек")

print(f"\nFine-tuning аяқталды. Жалпы уақыт: {time.time() - t_train:.1f} сек")

if SAVE_LORA:
    Path(LORA_SAVE_DIR).mkdir(parents=True, exist_ok=True)
    qwen_model.save_pretrained(LORA_SAVE_DIR)
    tokenizer.save_pretrained(LORA_SAVE_DIR)
    print(f"LoRA adapter сақталды: {LORA_SAVE_DIR}")

torch.cuda.empty_cache()


# ============================================================
# 16) PRECOMPUTE EMBEDDINGS AFTER FINE-TUNING
# ============================================================

qwen_model.eval()

train_questions = train_df["q_dense"].tolist()
test_questions = test_df["q_dense"].tolist()

train_answers = train_df["a_final"].tolist()
test_answers = test_df["a_final"].tolist()

train_q_tok_sets = train_df["q_tok_set"].tolist()
test_q_tok_sets = test_df["q_tok_set"].tolist()

train_q_seg_sets = train_df["q_seg_set"].tolist()
test_q_seg_sets = test_df["q_seg_set"].tolist()

print("\n3-қадам: Fine-tuned Qwen2.5 арқылы question embeddings есептелуде...")

t0 = time.time()

train_q_emb = qwen_encode(
    train_questions,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

test_q_emb = qwen_encode(
    test_questions,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

print(f"Question embeddings дайын: {time.time() - t0:.2f} сек")


print("\n4-қадам: Fine-tuned Qwen2.5 арқылы answer embeddings есептелуде...")

t0 = time.time()

train_a_emb = qwen_encode(
    train_answers,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

test_a_emb = qwen_encode(
    test_answers,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

print(f"Answer embeddings дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 17) HYBRID RETRIEVAL HELPERS
# ============================================================

def jaccard_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return len(a & b) / max(1, len(a | b))


def dice_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return (2.0 * len(a & b)) / max(1.0, len(a) + len(b))


def morph_similarity(
    q_tok_set: Set[str],
    c_tok_set: Set[str],
    q_seg_set: Set[str],
    c_seg_set: Set[str],
) -> float:
    token_score = jaccard_sim(q_tok_set, c_tok_set)
    seg_score = dice_sim(q_seg_set, c_seg_set)

    return 0.65 * token_score + 0.35 * seg_score


def cosine01(x: float) -> float:
    return max(0.0, min(1.0, (x + 1.0) * 0.5))


def get_topk_sorted(
    scores: np.ndarray,
    k: int
) -> Tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[0])

    if k <= 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx = np.argpartition(-scores, kth=k - 1)[:k]
    vals = scores[idx]
    order = np.argsort(-vals)

    return idx[order].astype(np.int32), vals[order].astype(np.float32)


def rerank_by_morph(
    q_tok_set: Set[str],
    q_seg_set: Set[str],
    cand_indices: List[int],
    cand_dense_scores: List[float],
    kb_q_tok_sets: List[Set[str]],
    kb_q_seg_sets: List[Set[str]],
):
    rows = []

    for idx, dscore in zip(cand_indices, cand_dense_scores):
        morph = morph_similarity(
            q_tok_set=q_tok_set,
            c_tok_set=kb_q_tok_sets[idx],
            q_seg_set=q_seg_set,
            c_seg_set=kb_q_seg_sets[idx],
        )

        dense01 = cosine01(float(dscore))

        score = (
            HYBRID_DENSE_WEIGHT * dense01
            +
            HYBRID_MORPH_WEIGHT * morph
        )

        rows.append({
            "idx": int(idx),
            "dense_qsim": float(dscore),
            "morph_sim": float(morph),
            "rerank_score": float(score),
        })

    rows.sort(key=lambda x: x["rerank_score"], reverse=True)
    return rows[:TOPK_EVAL]


# ============================================================
# 18) RETRIEVAL
# ============================================================

_eval_cache = {}


def precompute_eval_predictions():
    if "pred_answers" in _eval_cache:
        return

    print("\n5-қадам: TEST -> TRAIN retrieval есептелуде...")
    t0 = time.time()

    pred_indices = []
    pred_q_sims = []
    pred_morph_sims = []
    pred_rerank_scores = []
    pred_answers = []

    topk_indices_list = []
    topk_qsims_list = []
    topk_morph_list = []
    topk_rerank_list = []
    topk_answers_list = []

    n_test = len(test_q_emb)
    dense_topk = min(DENSE_TOPK, train_q_emb.shape[0])
    k_eval = min(TOPK_EVAL, train_q_emb.shape[0])

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]

        sims = batch @ train_q_emb.T

        local_top_idx = np.argpartition(
            -sims,
            kth=dense_topk - 1,
            axis=1
        )[:, :dense_topk]

        row_ids = np.arange(end - start)[:, None]

        local_top_vals = sims[
            row_ids,
            local_top_idx
        ]

        order = np.argsort(-local_top_vals, axis=1)

        local_top_idx = np.take_along_axis(
            local_top_idx,
            order,
            axis=1
        )

        local_top_vals = np.take_along_axis(
            local_top_vals,
            order,
            axis=1
        )

        for r in range(end - start):
            global_i = start + r

            if USE_MORPH_RERANK:
                rows = rerank_by_morph(
                    q_tok_set=test_q_tok_sets[global_i],
                    q_seg_set=test_q_seg_sets[global_i],
                    cand_indices=local_top_idx[r].tolist(),
                    cand_dense_scores=local_top_vals[r].tolist(),
                    kb_q_tok_sets=train_q_tok_sets,
                    kb_q_seg_sets=train_q_seg_sets,
                )
            else:
                rows = []

                for j in range(k_eval):
                    rows.append({
                        "idx": int(local_top_idx[r][j]),
                        "dense_qsim": float(local_top_vals[r][j]),
                        "morph_sim": 0.0,
                        "rerank_score": float(local_top_vals[r][j]),
                    })

            best = rows[0]
            best_idx = int(best["idx"])

            pred_indices.append(best_idx)
            pred_q_sims.append(float(best["dense_qsim"]))
            pred_morph_sims.append(float(best["morph_sim"]))
            pred_rerank_scores.append(float(best["rerank_score"]))
            pred_answers.append(train_answers[best_idx])

            row_indices = [
                int(x["idx"])
                for x in rows[:TOPK_EVAL]
            ]

            row_qsims = [
                float(x["dense_qsim"])
                for x in rows[:TOPK_EVAL]
            ]

            row_morphs = [
                float(x["morph_sim"])
                for x in rows[:TOPK_EVAL]
            ]

            row_reranks = [
                float(x["rerank_score"])
                for x in rows[:TOPK_EVAL]
            ]

            row_answers = [
                train_answers[i]
                for i in row_indices
            ]

            while len(row_indices) < TOPK_EVAL:
                row_indices.append(-1)
                row_qsims.append(0.0)
                row_morphs.append(0.0)
                row_reranks.append(0.0)
                row_answers.append("")

            topk_indices_list.append(row_indices)
            topk_qsims_list.append(row_qsims)
            topk_morph_list.append(row_morphs)
            topk_rerank_list.append(row_reranks)
            topk_answers_list.append(row_answers)

    _eval_cache["pred_indices"] = np.array(pred_indices, dtype=np.int32)
    _eval_cache["pred_q_sims"] = np.array(pred_q_sims, dtype=np.float32)
    _eval_cache["pred_morph_sims"] = np.array(pred_morph_sims, dtype=np.float32)
    _eval_cache["pred_rerank_scores"] = np.array(pred_rerank_scores, dtype=np.float32)
    _eval_cache["pred_answers"] = pred_answers
    _eval_cache["true_answers"] = test_answers
    _eval_cache["test_questions"] = [
        clean_text(q)
        for q in test_questions
    ]

    _eval_cache["topk_indices"] = np.array(topk_indices_list, dtype=np.int32)
    _eval_cache["topk_qsims"] = np.array(topk_qsims_list, dtype=np.float32)
    _eval_cache["topk_morph_sims"] = np.array(topk_morph_list, dtype=np.float32)
    _eval_cache["topk_rerank_scores"] = np.array(topk_rerank_list, dtype=np.float32)
    _eval_cache["topk_answers"] = topk_answers_list

    print(f"Retrieval дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 19) ANSWER SEMANTIC METRICS
# ============================================================

def prepare_answer_semantic_cache():
    if "ans_cos" in _eval_cache and "topk_ans_cos" in _eval_cache:
        return

    precompute_eval_predictions()

    pred_indices = _eval_cache["pred_indices"]
    topk_indices = _eval_cache["topk_indices"]

    pred_a_emb = train_a_emb[pred_indices]
    true_a_emb = test_a_emb

    ans_cos = np.sum(
        pred_a_emb * true_a_emb,
        axis=1
    )

    topk_a_emb = train_a_emb[topk_indices]

    topk_ans_cos = np.sum(
        topk_a_emb * true_a_emb[:, None, :],
        axis=2
    )

    _eval_cache["ans_cos"] = ans_cos.astype(np.float32)
    _eval_cache["topk_ans_cos"] = topk_ans_cos.astype(np.float32)


# ============================================================
# 20) BERTScore
# ============================================================

def compute_bertscore_f1_at_1():
    if "bertscore_itemwise" in _eval_cache and _eval_cache["bertscore_itemwise"] is not None:
        itemwise = _eval_cache["bertscore_itemwise"]
        return float(np.mean(itemwise)), itemwise

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n6-қадам: BERTScoreF1@1 есептелуде...")
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)

    t0 = time.time()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        _, _, f1 = bert_score(
            preds,
            trues,
            model_type=BERTSCORE_MODEL_TYPE,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    except Exception as e:
        print("Бірінші BERTScore есептеу сәтсіз болды:", repr(e))
        print("Fallback: lang='kk' арқылы есептеу басталды...")

        _, _, f1 = bert_score(
            preds,
            trues,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    if hasattr(f1, "detach"):
        itemwise = f1.detach().cpu().numpy().astype(float)
    else:
        itemwise = np.array(f1).astype(float)

    _eval_cache["bertscore_itemwise"] = itemwise
    _eval_cache["bertscore_f1_mean"] = float(np.mean(itemwise))
    _eval_cache["bertscore_model_type"] = BERTSCORE_MODEL_TYPE

    print(f"BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")

    return float(np.mean(itemwise)), itemwise


# ============================================================
# 21) METRICS
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        float(norm_for_exact(p) == norm_for_exact(t))
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_top3_exact_hit():
    precompute_eval_predictions()

    topk_answers = _eval_cache["topk_answers"]
    trues = _eval_cache["true_answers"]

    hits = []

    for row_answers, gold in zip(topk_answers, trues):
        hit = any(
            norm_for_exact(a) == norm_for_exact(gold)
            for a in row_answers
        )

        hits.append(float(hit))

    return float(np.mean(hits)), hits


def evaluate_token_f1_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        token_f1(p, t)
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()

    vals = _eval_cache["pred_q_sims"].astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_semantic_at_1_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    ans_cos = _eval_cache["ans_cos"]
    vals = (ans_cos >= threshold).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_recall_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    vals = np.any(topk_ans_cos >= threshold, axis=1).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_mrr_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    flags = (topk_ans_cos >= threshold).astype(np.int32)

    rr = []

    for row in flags:
        val = 0.0

        for rank, flag in enumerate(row.tolist(), start=1):
            if flag == 1:
                val = 1.0 / rank
                break

        rr.append(val)

    return float(np.mean(rr)), rr


# ============================================================
# 22) BOOTSTRAP CI
# ============================================================

def bootstrap_ci_mean(
    values: List[float],
    n_boot: int = BOOT_N,
    seed: int = SEED
) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    n = len(arr)

    if n == 0:
        return float("nan"), float("nan")

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = float(np.mean(arr[idx]))

    lo = float(np.percentile(boots, 2.5))
    hi = float(np.percentile(boots, 97.5))

    return lo, hi


def fmt_metric(name: str, mean_val, vals=None):
    if RUN_BOOTSTRAP_CI and vals is not None:
        lo, hi = bootstrap_ci_mean(vals)
        print(f"{name:<34}: {mean_val:.6f} | 95% CI [{lo:.6f}, {hi:.6f}]")
    else:
        print(f"{name:<34}: {mean_val:.6f}")


# ============================================================
# 23) DETAILS CSV
# ============================================================

def build_eval_details():
    precompute_eval_predictions()
    prepare_answer_semantic_cache()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    questions = _eval_cache["test_questions"]

    qsims = _eval_cache["pred_q_sims"]
    msims = _eval_cache["pred_morph_sims"]
    rscores = _eval_cache["pred_rerank_scores"]

    topk_indices = _eval_cache["topk_indices"]
    topk_qsims = _eval_cache["topk_qsims"]
    topk_morph_sims = _eval_cache["topk_morph_sims"]
    topk_rerank_scores = _eval_cache["topk_rerank_scores"]
    topk_answers = _eval_cache["topk_answers"]

    ans_cos = _eval_cache["ans_cos"]
    topk_ans_cos = _eval_cache["topk_ans_cos"]

    if RUN_BERTSCORE:
        bert_mean, bert_itemwise = compute_bertscore_f1_at_1()
    else:
        bert_mean, bert_itemwise = None, None

    rows = []

    for i, (q, gold, pred) in enumerate(zip(questions, trues, preds)):
        top_answers = topk_answers[i]

        top_exact_flags = [
            float(norm_for_exact(a) == norm_for_exact(gold))
            for a in top_answers
        ]

        top_sem_flags = [
            float(x >= SEM_THR)
            for x in topk_ans_cos[i].tolist()
        ]

        recall3 = float(any(top_sem_flags))

        mrr3 = 0.0

        for rank, flag in enumerate(top_sem_flags, start=1):
            if flag == 1.0:
                mrr3 = 1.0 / rank
                break

        row = {
            "item_key": stable_item_key(q, gold),
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,

            "QSim": float(qsims[i]),
            "MorphSim": float(msims[i]),
            "RerankScore": float(rscores[i]),

            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "Top3ExactHit": float(any(top_exact_flags)),
            "TokenF1": float(token_f1(pred, gold)),

            "AnsCos": float(ans_cos[i]),
            "SemHit": float(ans_cos[i] >= SEM_THR),
            "Recall@3": recall3,
            "MRR@3": float(mrr3),

            "Top3_indices": json.dumps(topk_indices[i].tolist(), ensure_ascii=False),
            "Top3_QSims": json.dumps(
                [round(float(x), 6) for x in topk_qsims[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_MorphSims": json.dumps(
                [round(float(x), 6) for x in topk_morph_sims[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_RerankScores": json.dumps(
                [round(float(x), 6) for x in topk_rerank_scores[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_pred_answers": json.dumps(top_answers, ensure_ascii=False),
            "Top3_AnsCos": json.dumps(
                [round(float(x), 6) for x in topk_ans_cos[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_ExactFlags": json.dumps(top_exact_flags, ensure_ascii=False),
            "Top3_SemFlags": json.dumps(top_sem_flags, ensure_ascii=False),
        }

        if bert_itemwise is not None:
            row["BERTScoreF1"] = float(bert_itemwise[i])

        rows.append(row)

    details_df = pd.DataFrame(rows)

    return details_df


def export_details_csv(path: str):
    details_df = build_eval_details()
    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"CSV сақталды: {path} | rows={len(details_df)}")
    return details_df


# ============================================================
# 24) FINAL EVALUATION
# ============================================================

def evaluate_femseg_mapp_finetuned_qwen_retrieval():
    t0 = time.time()

    precompute_eval_predictions()

    exact, exact_vals = evaluate_exact_at_1()
    top3_exact, top3_exact_vals = evaluate_top3_exact_hit()
    token_f1_mean, token_f1_vals = evaluate_token_f1_at_1()
    qsim_mean, qsim_vals = evaluate_mean_cos_qsim_at_1()

    sem1, sem1_vals = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    recall3, recall3_vals = evaluate_recall_at_3_ans_cos(threshold=SEM_THR)
    mrr3, mrr3_vals = evaluate_mrr_at_3_ans_cos(threshold=SEM_THR)

    if RUN_BERTSCORE:
        bertscore_f1, bertscore_vals = compute_bertscore_f1_at_1()
    else:
        bertscore_f1, bertscore_vals = None, None

    if EXPORT_CSV:
        details_df = export_details_csv(CSV_PATH)
    else:
        details_df = build_eval_details()

    print("\n==================== FINE-TUNED FULL RESULTS ====================")
    print("Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 fine-tuning -> Retrieval")
    print(f"QWEN_MODEL_NAME              : {QWEN_MODEL_NAME}")
    print(f"Dataset rows used            : {len(qa_data)}")
    print(f"Train rows                   : {len(train_df)}")
    print(f"Test rows                    : {len(test_answers)}")
    print(f"Epochs                       : {NUM_EPOCHS}")
    print(f"Effective batch size          : {EFFECTIVE_BATCH_SIZE}")
    print(f"Learning rate                 : {LEARNING_RATE}")
    print(f"SEM_THR                       : {SEM_THR}")
    print(f"TOPK_EVAL                     : {TOPK_EVAL}")
    print(f"USE_MORPH_RERANK              : {USE_MORPH_RERANK}")

    fmt_metric("Exact@1", exact, exact_vals)
    fmt_metric("Top-3 ExactHit", top3_exact, top3_exact_vals)
    fmt_metric("TokenF1@1", token_f1_mean, token_f1_vals)
    fmt_metric("MeanCos@1(QSim)", qsim_mean, qsim_vals)

    fmt_metric(f"Semantic@1(ans_cos>={SEM_THR})", sem1, sem1_vals)
    fmt_metric(f"Recall@3(ans_cos>={SEM_THR})", recall3, recall3_vals)
    fmt_metric(f"MRR@3(ans_cos>={SEM_THR})", mrr3, mrr3_vals)

    if bertscore_f1 is not None:
        fmt_metric("BERTScoreF1@1", bertscore_f1, bertscore_vals)
        print(f"BERTScore model              : {_eval_cache.get('bertscore_model_type')}")
    else:
        print("BERTScoreF1@1                : N/A")

    print(f"Evaluation time              : {time.time() - t0:.2f} sec")
    print("===============================================================")

    return details_df


# ============================================================
# 25) OPTIONAL INTERACTIVE QA
# ============================================================

_query_emb_cache = {}


def build_runtime_query_views(question: str):
    q_clean = clean_text(question)
    q_sp = femseg_to_sp(q_clean)
    q_mapp = mapp_normalize(q_clean)

    q_dense = compose_dense_view(
        q_clean=q_clean,
        q_sp=q_sp,
        q_mapp=q_mapp
    )

    q_tok_set = set(WORD_RE.findall(q_mapp))
    q_seg_set = sp_token_set(q_sp)

    return q_clean, q_sp, q_mapp, q_dense, q_tok_set, q_seg_set


def get_query_embedding(question: str) -> Tuple[np.ndarray, Set[str], Set[str], str, str, str, str]:
    q_clean, q_sp, q_mapp, q_dense, q_tok_set, q_seg_set = build_runtime_query_views(question)

    if q_dense in _query_emb_cache:
        return _query_emb_cache[q_dense], q_tok_set, q_seg_set, q_clean, q_sp, q_mapp, q_dense

    qv = qwen_encode(
        [q_dense],
        batch_size=1,
        normalize=True,
        max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
        show_progress=False
    )

    _query_emb_cache[q_dense] = qv

    return qv, q_tok_set, q_seg_set, q_clean, q_sp, q_mapp, q_dense


def ask_question_runtime(question: str):
    qv, q_tok_set, q_seg_set, q_clean, q_sp, q_mapp, q_dense = get_query_embedding(question)

    sims = (train_q_emb @ qv.T).squeeze(1)

    top_idx, top_vals = get_topk_sorted(
        sims,
        DENSE_TOPK
    )

    if USE_MORPH_RERANK:
        rows = rerank_by_morph(
            q_tok_set=q_tok_set,
            q_seg_set=q_seg_set,
            cand_indices=top_idx.tolist(),
            cand_dense_scores=top_vals.tolist(),
            kb_q_tok_sets=train_q_tok_sets,
            kb_q_seg_sets=train_q_seg_sets,
        )
    else:
        rows = []

        for j in range(min(TOPK_EVAL, len(top_idx))):
            rows.append({
                "idx": int(top_idx[j]),
                "dense_qsim": float(top_vals[j]),
                "morph_sim": 0.0,
                "rerank_score": float(top_vals[j]),
            })

    best = rows[0]
    best_idx = int(best["idx"])
    best_answer = train_answers[best_idx]

    return (
        best_answer,
        best_idx,
        float(best["dense_qsim"]),
        float(best["morph_sim"]),
        float(best["rerank_score"]),
        q_clean,
        q_sp,
        q_mapp,
        q_dense,
    )


RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    print("\nИнтерактив режим қосылды.")
    print("Шығу үшін exit деп жазыңыз.")

    while True:
        user_input = input("\nСұрақ енгізіңіз: ")

        if user_input.strip().lower() == "exit":
            break

        (
            answer,
            idx,
            qsim,
            morph,
            rerank,
            q_clean,
            q_sp,
            q_mapp,
            q_dense,
        ) = ask_question_runtime(user_input)

        print("\n================ QA ЖАУАП ================")
        print(answer)

        print("\n---------------- DETAILS ----------------")
        print(f"idx          : {idx}")
        print(f"qsim         : {qsim:.4f}")
        print(f"morph        : {morph:.4f}")
        print(f"rerank       : {rerank:.4f}")

        print("\n---------------- QUERY VIEWS ----------------")
        print("clean        :", q_clean)
        print("FEMSeg       :", q_sp)
        print("MAPP         :", q_mapp)
        print("model input  :", q_dense)


# ============================================================
# 26) RUN FINAL EVALUATION
# ============================================================

details_df = evaluate_femseg_mapp_finetuned_qwen_retrieval()
display(details_df.head(5))

DEVICE: cuda
Mounted at /content/drive
QA JSON файл: grok_python_dataset.json
Барлық QA саны: 5944


,question,answer
0,Python дегеніміз не?,Python - 1991 жылы Гвидо ван Россум жасаған қа...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты -...
2,Python - бағдарламалау тілі қандай салаларда ...,Python - бағдарламалау тілі көптеген салаларда...


FEMSeg жүктелуде...
FEMSeg дайын.
FEMSeg тест: ▁Мен ▁мектеп ке ▁бар а мын

1-қадам: clean + FEMSeg + MAPP дайындалуда...

FEMSeg segmentation басталды...
FEMSeg 1000/5537 дайын | 1.2 сек
FEMSeg 2000/5537 дайын | 2.3 сек
FEMSeg 3000/5537 дайын | 3.6 сек
FEMSeg 4000/5537 дайын | 4.5 сек
FEMSeg 5000/5537 дайын | 5.4 сек
FEMSeg 5537/5537 дайын | 6.2 сек

MAPP normalization басталды...
MAPP 1000/5537 дайын | 0.3 сек
MAPP 2000/5537 дайын | 0.6 сек
MAPP 3000/5537 дайын | 0.8 сек
MAPP 4000/5537 дайын | 1.1 сек
MAPP 5000/5537 дайын | 1.4 сек
MAPP 5537/5537 дайын | 1.7 сек

Дайын preprocessing мысалы:


,question,q_clean,q_sp,q_mapp,q_dense,a_final
0,Python дегеніміз не?,Python дегеніміз не?,▁Python ▁де ген і міз ▁не?,python де не,Python дегеніміз не? [FEMSEG] ▁Python ▁де ген ...,Python-1991 жылы Гвидо ван Россум жасаған қара...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты қ...,▁Python ▁бағдарламалау ▁тіл і нің ▁негізгі ▁ма...,python бағдарламалау тіл негізгі мақсаты қандай,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты-а...
2,Python - бағдарламалау тілі қандай салаларда ...,Python-бағдарламалау тілі қандай салаларда қол...,▁Python-бағдарламалау ▁тілі ▁қандай ▁сала лар ...,python-бағдарламалау тілі қандай сала қолдан,Python-бағдарламалау тілі қандай салаларда қол...,Python-бағдарламалау тілі көптеген салаларда қ...



TRAIN: 5349 | TEST: 595 | seed=42

2-қадам: Qwen2.5 model жүктеледі.
Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 fine-tuning -> Retrieval
QWEN_MODEL_NAME: Qwen/Qwen2.5-1.5B-Instruct


...

  0%|          | 0/68 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/38 [00:00<?, ?it/s]

done in 2.66 seconds, 223.57 sentences/sec
BERTScoreF1@1 дайын: 6.95 сек
CSV сақталды: femseg_mapp_qwen25_finetuned_full_metrics.csv | rows=595

==================== FINE-TUNED FULL RESULTS ====================
Pipeline: Dataset -> FEMSeg -> MAPP -> q_dense -> Qwen2.5 fine-tuning -> Retrieval
QWEN_MODEL_NAME              : Qwen/Qwen2.5-1.5B-Instruct
Dataset rows used            : 5944
Train rows                   : 5349
Test rows                    : 595
Epochs                       : 2
Effective batch size          : 16
Learning rate                 : 0.0002
SEM_THR                       : 0.85
TOPK_EVAL                     : 3
USE_MORPH_RERANK              : True
Exact@1                           : 0.090756
Top-3 ExactHit                    : 0.102521
TokenF1@1                         : 0.456868
MeanCos@1(QSim)                   : 0.878376
Semantic@1(ans_cos>=0.85)         : 0.373109
Recall@3(ans_cos>=0.85)           : 0.445378
MRR@3(ans_cos>=0.85)              : 0.405322
BERTScoreF1

,item_key,test_question,gold_answer,pred_answer,QSim,MorphSim,RerankScore,Exact,Top3ExactHit,TokenF1,...,MRR@3,Top3_indices,Top3_QSims,Top3_MorphSims,Top3_RerankScores,Top3_pred_answers,Top3_AnsCos,Top3_ExactFlags,Top3_SemFlags,BERTScoreF1
0,688a070c23278fcb3a801c344c614754,Сөздіктегі кілт-мағына жұбын қалай жоямыз? [FE...,pop() әдісін қолданамыз. Мысалы: sozdik = {'a'...,Кілт арқылы мән тағайындаймыз. Мысалы: sozdik ...,0.948972,0.666667,0.897531,0.0,0.0,0.580645,...,0.0,"[508, 2273, 3646]","[0.948972, 0.963146, 0.934068]","[0.666667, 0.473016, 0.291813]","[0.897531, 0.854434, 0.798229]","[""Кілт арқылы мән тағайындаймыз. Мысалы: sozdi...","[0.773707, 0.627505, 0.724665]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.862339
1,185712d8a3607e4142f1e73f4b38ced3,GitHub Actions-те орналастыруды қалай автоматт...,"AWS, Azure немесе SSH арқылы серверге кодты жі...",actions/upload-artifact немесе rsync арқылы се...,0.809933,0.478571,0.798368,0.0,0.0,0.500000,...,0.0,"[1341, 2565, 1094]","[0.809933, 0.813126, 0.797461]","[0.478571, 0.465238, 0.465238]","[0.798368, 0.796232, 0.790357]","[""actions/upload-artifact немесе rsync арқылы ...","[0.665858, 0.692564, 0.552991]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.831136
2,b48c12089d7ac8af81b26112952876aa,Боттың қолжетімділігін қалай мониторинг жасайм...,UptimeRobot немесе custom logging арқылы.,GitHub Actions арқылы Markdown синтаксисін тек...,0.773701,0.525000,0.796388,0.0,0.0,0.181818,...,0.0,"[4984, 1666, 3705]","[0.773701, 0.666609, 0.749071]","[0.525, 0.605, 0.453571]","[0.796388, 0.776228, 0.769294]","[""GitHub Actions арқылы Markdown синтаксисін т...","[0.394926, 0.724673, 0.716399]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.718941
3,8e7ec71376dd621b014588074c3718b5,Функцияда қалай прогресс барын көрсетеміз? [FE...,tqdm модулін қолданамыз. Мысалы: from tqdm imp...,cProfile модулін қолданамыз. Мысалы: import cP...,0.453461,0.343939,0.631033,0.0,0.0,0.470588,...,0.0,"[1392, 4852, 3918]","[0.453461, 0.407622, 0.391458]","[0.343939, 0.360714, 0.3125]","[0.631033, 0.618037, 0.599922]","[""cProfile модулін қолданамыз. Мысалы: import ...","[0.240096, 0.209177, 0.272294]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.814221
4,5e50a8e61907c935033018151f3e8c86,GET пен POST сұрауларының басты айырмашылығы н...,"GET деректерді алады, POST деректерді жібереді...","Әдетте жоқ, өйткені олар серверде өзгеріс жаса...",0.692005,0.300000,0.709502,0.0,0.0,0.352941,...,0.0,"[3273, 3163, 1339]","[0.692005, 0.685881, 0.658369]","[0.3, 0.294706, 0.327193]","[0.709502, 0.705882, 0.703687]","[""Әдетте жоқ, өйткені олар серверде өзгеріс жа...","[0.710928, 0.737112, 0.717419]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.759743


Dataset -> Qwen2.5 -> Retrieval

In [ ]:
# @title Dataset -> Qwen2.5 -> Retrieval | Full Metrics | Google Colab Ready

# ============================================================
# GOOGLE COLAB READY CODE
#
# ҚАТАҢ САҚТАЛҒАН ЖАҢА ҚҰРЫЛЫМ:
#
# Dataset
#   -> Qwen2.5 embedding
#   -> Retrieval
#   -> Evaluation
#
# NO FEMSeg
# NO MAPP
# NO q_dense
# NO FINE-TUNING
#
# Metrics:
#   Exact@1
#   Top-3 ExactHit
#   TokenF1@1
#   MeanCos@1(QSim)
#   Semantic@1(ans_cos >= 0.85)
#   Recall@3(ans_cos >= 0.85)
#   MRR@3(ans_cos >= 0.85)
#   BERTScoreF1@1
# ============================================================


# ============================================================
# 0) INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers accelerate bitsandbytes bert-score


# ============================================================
# 1) IMPORTS AND ENVIRONMENT
# ============================================================

import os
os.environ["WANDB_DISABLED"] = "true"

import re
import glob
import time
import json
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from bert_score import score as bert_score

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 2) SPEED SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE != "cuda":
    raise RuntimeError(
        "Qwen2.5 үшін GPU керек. Colab-та Runtime -> Change runtime type -> GPU таңдаңыз."
    )


# ============================================================
# 3) CONFIG
# ============================================================

DATA_JSON_PATH = "grok_python_dataset.json"

TEST_SIZE = 0.10
SEM_THR = 0.85
TOPK_EVAL = 3

ENCODE_BATCH_SIZE = 8
SIM_BATCH_SIZE = 256

QWEN_MAX_LENGTH = 128

EXPORT_CSV = True
CSV_PATH = "dataset_qwen25_retrieval_full_metrics.csv"

USE_CACHE = True

# Жылдам нұсқа:
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Егер міндетті түрде 7B керек болса, жоғарыдағы жолды comment жасап, мынаны ашыңыз:
# QWEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# BERTScore
RUN_BERTSCORE = True
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
# Егер сапалырақ, бірақ ауыр нұсқа керек болса:
# BERTSCORE_MODEL_TYPE = "xlm-roberta-large"

BERTSCORE_BATCH_SIZE = 16

# Bootstrap керек болса True жасаңыз.
RUN_BOOTSTRAP_CI = False
BOOT_N = 500


# ============================================================
# 4) GOOGLE DRIVE AND CACHE
# ============================================================

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)

CACHE_DIR = Path("/content/drive/MyDrive/KAZ_MORPH/qwen_dataset_only_cache") if IN_COLAB else Path("./qwen_dataset_only_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 5) ROBUST JSON LOADING
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p

    candidates = [
        f"/content/{p}",
        f"/content/drive/MyDrive/{p}",
        f"/content/drive/MyDrive/KAZ_MORPH/{p}",
    ]

    for c in candidates:
        if Path(c).exists():
            return c

    name = Path(p).name
    hits = glob.glob(f"**/{name}", recursive=True)

    if hits:
        return hits[0]

    near = glob.glob("**/*.json", recursive=True)

    raise FileNotFoundError(
        f"Файл табылмады: {p}\n"
        f"Табылған .json файлдар:\n" + "\n".join(near[:30])
    )


def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)


def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)


def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text


def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None

    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    return None


def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)

    if normalized is not None:
        return [normalized]

    if isinstance(obj, list):
        out = []

        for item in obj:
            norm = _normalize_record(item)

            if norm is not None:
                out.append(norm)

        return out

    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                out = []

                for item in obj[key]:
                    norm = _normalize_record(item)

                    if norm is not None:
                        out.append(norm)

                return out

    return []


def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    obj = json.loads(_normalize_json_text(text))
    records = _extract_records(obj)

    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")

    return records


def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx = 0
    records = []

    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1

        if idx >= len(text):
            break

        obj, end = decoder.raw_decode(text, idx)
        idx = end

        extracted = _extract_records(obj)

        if extracted:
            records.extend(extracted)

    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")

    return records


def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records = []
    bad_lines = []

    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()

        if not s:
            continue

        if s.endswith(","):
            s = s[:-1].strip()

        s = _normalize_json_text(s)

        try:
            obj = json.loads(s)
            extracted = _extract_records(obj)

            if extracted:
                records.extend(extracted)
            else:
                bad_lines.append((line_no, "QA өрістері табылмады", s[:200]))

        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))

    if not records:
        preview = "\n".join(
            [f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]]
        )

        raise ValueError(
            "Файлдан бірде-бір QA жазба оқылмады.\n"
            f"Алғашқы қате жолдар:\n{preview}"
        )

    if bad_lines:
        print(f"Ескерту: {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")

    return records


def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []

    for parser in (
        _parse_as_single_json,
        _parse_as_multiple_json_objects,
        _parse_line_by_line,
    ):
        try:
            records = parser(text)

            if records:
                return records

        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")

    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


data_path = find_data_path(DATA_JSON_PATH)
print("QA JSON файл:", data_path)

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON ішінде question/answer немесе instruction/response болуы керек."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"] = qa_data["answer"].astype(str).str.strip()

qa_data = qa_data[
    (qa_data["question"] != "") &
    (qa_data["answer"] != "")
].reset_index(drop=True)

print("Барлық QA саны:", len(qa_data))
display(qa_data.head(3))


# ============================================================
# 6) BASIC TEXT PROCESSING AND METRICS
# ============================================================

_punct_space_left = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space = re.compile(r"\s+")


def clean_text(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_text(text).lower()).strip()


def tokens(text: str) -> List[str]:
    t = clean_text(text).lower()

    return re.findall(
        r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+",
        t
    )


def token_f1(pred: str, gold: str) -> float:
    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    pc = Counter(p)
    gc = Counter(g)

    inter = sum((pc & gc).values())

    if inter == 0:
        return 0.0

    precision = inter / max(1, len(p))
    recall = inter / max(1, len(g))

    return (2 * precision * recall) / (precision + recall + 1e-12)


def stable_item_key(test_question: str, gold_answer: str) -> str:
    raw = norm_for_exact(test_question) + " ||| " + norm_for_exact(gold_answer)
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


def make_cache_key(data_path: str, n_rows: int) -> str:
    p = Path(data_path)
    st = p.stat()
    raw = (
        f"{p.name}|{st.st_size}|{st.st_mtime_ns}|"
        f"rows={n_rows}|model={QWEN_MODEL_NAME}|maxlen={QWEN_MAX_LENGTH}"
    )
    return hashlib.md5(raw.encode("utf-8")).hexdigest()[:16]


# ============================================================
# 7) DATASET PREPARATION
# ============================================================
# ЖАҢА ҚҰРЫЛЫМ:
# Dataset -> Qwen2.5 -> Retrieval
#
# Мұнда FEMSeg жоқ.
# Мұнда MAPP жоқ.
# Мұнда q_dense жоқ.
#
# Модельге тікелей q_input беріледі.

print("\n1-қадам: Dataset input дайындалуда...")

qa_data["q_input"] = qa_data["question"].map(clean_text)
qa_data["a_final"] = qa_data["answer"].map(clean_text)

print("Dataset дайын.")
display(qa_data[["question", "q_input", "answer", "a_final"]].head(3))


# ============================================================
# 8) LOAD Qwen2.5 MODEL AFTER DATASET
# ============================================================

print("\n2-қадам: Qwen2.5 model жүктеледі.")
print("Pipeline: Dataset -> Qwen2.5 -> Retrieval")
print("QWEN_MODEL_NAME:", QWEN_MODEL_NAME)

qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME,
    trust_remote_code=True
)

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_tokenizer.padding_side = "right"

USE_4BIT = "7B" in QWEN_MODEL_NAME

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

else:
    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(DEVICE)

qwen_model.eval()

QWEN_DEVICE = next(qwen_model.parameters()).device

print("Qwen2.5 model дайын.")
print("QWEN_DEVICE:", QWEN_DEVICE)


# ============================================================
# 9) TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    qa_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_idx = train_df.index.to_numpy()
test_idx = test_df.index.to_numpy()

train_answers = train_df["a_final"].tolist()
test_answers = test_df["a_final"].tolist()
test_questions = test_df["q_input"].tolist()

print(f"\nTRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED}")


# ============================================================
# 10) Qwen2.5 EMBEDDING
# ============================================================

def _last_token_pool(
    last_hidden_state: torch.Tensor,
    attention_mask: torch.Tensor
) -> torch.Tensor:
    lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(
        last_hidden_state.size(0),
        device=last_hidden_state.device
    )
    pooled = last_hidden_state[batch_idx, lengths]
    return pooled


@torch.inference_mode()
def qwen_encode(
    texts: List[str],
    batch_size: int = ENCODE_BATCH_SIZE,
    normalize: bool = True,
    max_length: int = QWEN_MAX_LENGTH,
    show_progress: bool = True
) -> np.ndarray:
    all_vecs = []
    n = len(texts)
    t0 = time.time()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_texts = [str(x) for x in texts[start:end]]

        enc = qwen_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {
            k: v.to(QWEN_DEVICE)
            for k, v in enc.items()
        }

        out = qwen_model(
            **enc,
            output_hidden_states=False,
            return_dict=True
        )

        pooled = _last_token_pool(
            out.last_hidden_state,
            enc["attention_mask"]
        )

        if normalize:
            pooled = F.normalize(pooled, p=2, dim=1)

        all_vecs.append(
            pooled.detach().float().cpu().numpy()
        )

        if show_progress and ((end % 1000 == 0) or end == n):
            print(f"Encoded {end}/{n} | {time.time() - t0:.1f} сек")

    return np.vstack(all_vecs).astype(np.float32)


# ============================================================
# 11) PRECOMPUTE QUESTION EMBEDDINGS
# ============================================================

CACHE_KEY = make_cache_key(data_path, len(qa_data))
MODEL_SAFE_NAME = QWEN_MODEL_NAME.replace("/", "_").replace("-", "_")
Q_EMB_CACHE_PATH = CACHE_DIR / f"dataset_only_q_emb_{MODEL_SAFE_NAME}_{CACHE_KEY}.npy"

qa_q_input_full = qa_data["q_input"].tolist()
qa_a_final_full = qa_data["a_final"].tolist()

if USE_CACHE and Q_EMB_CACHE_PATH.exists():
    print(f"\nQuestion embeddings кэштен жүктелді: {Q_EMB_CACHE_PATH}")
    qa_q_emb_full = np.load(Q_EMB_CACHE_PATH)

else:
    print("\n3-қадам: Dataset questions -> Qwen2.5 embeddings есептелуде...")
    print("Model input: q_input = dataset question only")
    t0 = time.time()

    qa_q_emb_full = qwen_encode(
        qa_q_input_full,
        batch_size=ENCODE_BATCH_SIZE,
        normalize=True,
        max_length=QWEN_MAX_LENGTH,
        show_progress=True
    )

    print(f"Question embeddings дайын: {time.time() - t0:.2f} сек")

    if USE_CACHE:
        np.save(Q_EMB_CACHE_PATH, qa_q_emb_full)
        print(f"Question embeddings кэшке сақталды: {Q_EMB_CACHE_PATH}")

train_q_emb = qa_q_emb_full[train_idx]
test_q_emb = qa_q_emb_full[test_idx]


# ============================================================
# 12) RETRIEVAL
# ============================================================

_query_emb_cache = {}


def get_query_embedding(question: str) -> np.ndarray:
    q = clean_text(question)

    if q in _query_emb_cache:
        return _query_emb_cache[q]

    qv = qwen_encode(
        [q],
        batch_size=1,
        normalize=True,
        max_length=QWEN_MAX_LENGTH,
        show_progress=False
    )

    _query_emb_cache[q] = qv

    return qv


def get_topk_sorted(
    scores: np.ndarray,
    k: int
) -> Tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[0])

    if k <= 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx = np.argpartition(-scores, kth=k - 1)[:k]
    vals = scores[idx]
    order = np.argsort(-vals)

    return idx[order].astype(np.int32), vals[order].astype(np.float32)


def retrieve_from_kb(
    question: str,
    kb_q_emb: np.ndarray,
    kb_answers: List[str],
    topk: int = TOPK_EVAL,
):
    qv = get_query_embedding(question)
    sims = (kb_q_emb @ qv.T).squeeze(1)

    top_idx, top_vals = get_topk_sorted(sims, topk)

    best_idx = int(top_idx[0])
    best_sim = float(top_vals[0])
    best_answer = kb_answers[best_idx]

    return best_answer, best_idx, best_sim, top_idx, top_vals


def ask_question_runtime(question: str):
    return retrieve_from_kb(
        question=question,
        kb_q_emb=qa_q_emb_full,
        kb_answers=qa_a_final_full,
        topk=TOPK_EVAL,
    )


# ============================================================
# 13) EVALUATION CACHE
# ============================================================

_eval_cache = {}


def precompute_eval_predictions():
    if "pred_answers" in _eval_cache:
        return

    print("\n4-қадам: TEST -> TRAIN retrieval есептелуде...")
    t0 = time.time()

    pred_indices = []
    pred_q_sims = []
    pred_answers = []

    topk_indices_list = []
    topk_qsims_list = []
    topk_answers_list = []

    n_test = len(test_q_emb)
    k_eval = min(TOPK_EVAL, train_q_emb.shape[0])

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]

        sims = batch @ train_q_emb.T

        local_top_idx = np.argpartition(
            -sims,
            kth=k_eval - 1,
            axis=1
        )[:, :k_eval]

        row_ids = np.arange(end - start)[:, None]

        local_top_vals = sims[
            row_ids,
            local_top_idx
        ]

        order = np.argsort(-local_top_vals, axis=1)

        local_top_idx = np.take_along_axis(
            local_top_idx,
            order,
            axis=1
        )

        local_top_vals = np.take_along_axis(
            local_top_vals,
            order,
            axis=1
        )

        for r in range(end - start):
            row_indices = local_top_idx[r].astype(int).tolist()
            row_sims = local_top_vals[r].astype(float).tolist()
            row_answers = [train_answers[i] for i in row_indices]

            pred_indices.append(row_indices[0])
            pred_q_sims.append(row_sims[0])
            pred_answers.append(row_answers[0])

            topk_indices_list.append(row_indices)
            topk_qsims_list.append(row_sims)
            topk_answers_list.append(row_answers)

    _eval_cache["pred_indices"] = np.array(pred_indices, dtype=np.int32)
    _eval_cache["pred_q_sims"] = np.array(pred_q_sims, dtype=np.float32)
    _eval_cache["pred_answers"] = pred_answers
    _eval_cache["true_answers"] = test_answers
    _eval_cache["test_questions"] = [clean_text(q) for q in test_questions]
    _eval_cache["topk_indices"] = np.array(topk_indices_list, dtype=np.int32)
    _eval_cache["topk_qsims"] = np.array(topk_qsims_list, dtype=np.float32)
    _eval_cache["topk_answers"] = topk_answers_list

    print(f"Retrieval дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 14) ANSWER SEMANTIC METRICS
# ============================================================

def prepare_answer_semantic_cache():
    if "ans_cos" in _eval_cache and "topk_ans_cos" in _eval_cache:
        return

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    topk_answers = _eval_cache["topk_answers"]

    print("\n5-қадам: Answer semantic embeddings есептелуде...")
    t0 = time.time()

    pred_emb = qwen_encode(
        preds,
        batch_size=ENCODE_BATCH_SIZE,
        normalize=True,
        max_length=QWEN_MAX_LENGTH,
        show_progress=True
    )

    true_emb = qwen_encode(
        trues,
        batch_size=ENCODE_BATCH_SIZE,
        normalize=True,
        max_length=QWEN_MAX_LENGTH,
        show_progress=True
    )

    ans_cos = np.sum(pred_emb * true_emb, axis=1)

    flat_topk = []

    for row in topk_answers:
        flat_topk.extend(row)

    topk_emb = qwen_encode(
        flat_topk,
        batch_size=ENCODE_BATCH_SIZE,
        normalize=True,
        max_length=QWEN_MAX_LENGTH,
        show_progress=True
    )

    topk_emb = topk_emb.reshape(len(trues), TOPK_EVAL, -1)
    topk_ans_cos = np.sum(topk_emb * true_emb[:, None, :], axis=2)

    _eval_cache["ans_cos"] = ans_cos.astype(np.float32)
    _eval_cache["topk_ans_cos"] = topk_ans_cos.astype(np.float32)

    print(f"Answer semantic embeddings дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 15) BERTScore
# ============================================================

def compute_bertscore_f1_at_1():
    if "bertscore_itemwise" in _eval_cache and _eval_cache["bertscore_itemwise"] is not None:
        itemwise = _eval_cache["bertscore_itemwise"]
        return float(np.mean(itemwise)), itemwise

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n6-қадам: BERTScoreF1@1 есептелуде...")
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)

    t0 = time.time()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        _, _, f1 = bert_score(
            preds,
            trues,
            model_type=BERTSCORE_MODEL_TYPE,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    except Exception as e:
        print("Бірінші BERTScore есептеу сәтсіз болды:", repr(e))
        print("Fallback: lang='kk' арқылы есептеу басталды...")

        _, _, f1 = bert_score(
            preds,
            trues,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    if hasattr(f1, "detach"):
        itemwise = f1.detach().cpu().numpy().astype(float)
    else:
        itemwise = np.array(f1).astype(float)

    _eval_cache["bertscore_itemwise"] = itemwise
    _eval_cache["bertscore_f1_mean"] = float(np.mean(itemwise))
    _eval_cache["bertscore_model_type"] = BERTSCORE_MODEL_TYPE

    print(f"BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")

    return float(np.mean(itemwise)), itemwise


# ============================================================
# 16) METRICS
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        float(norm_for_exact(p) == norm_for_exact(t))
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_top3_exact_hit():
    precompute_eval_predictions()

    topk_answers = _eval_cache["topk_answers"]
    trues = _eval_cache["true_answers"]

    hits = []

    for row_answers, gold in zip(topk_answers, trues):
        hit = any(
            norm_for_exact(a) == norm_for_exact(gold)
            for a in row_answers
        )

        hits.append(float(hit))

    return float(np.mean(hits)), hits


def evaluate_token_f1_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        token_f1(p, t)
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()

    vals = _eval_cache["pred_q_sims"].astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_semantic_at_1_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    ans_cos = _eval_cache["ans_cos"]
    vals = (ans_cos >= threshold).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_recall_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    vals = np.any(topk_ans_cos >= threshold, axis=1).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_mrr_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    flags = (topk_ans_cos >= threshold).astype(np.int32)

    rr = []

    for row in flags:
        val = 0.0

        for rank, flag in enumerate(row.tolist(), start=1):
            if flag == 1:
                val = 1.0 / rank
                break

        rr.append(val)

    return float(np.mean(rr)), rr


# ============================================================
# 17) BOOTSTRAP CI
# ============================================================

def bootstrap_ci_mean(
    values: List[float],
    n_boot: int = BOOT_N,
    seed: int = SEED
) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    n = len(arr)

    if n == 0:
        return float("nan"), float("nan")

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = float(np.mean(arr[idx]))

    lo = float(np.percentile(boots, 2.5))
    hi = float(np.percentile(boots, 97.5))

    return lo, hi


def fmt_metric(name: str, mean_val, vals=None):
    if RUN_BOOTSTRAP_CI and vals is not None:
        lo, hi = bootstrap_ci_mean(vals)
        print(f"{name:<34}: {mean_val:.6f} | 95% CI [{lo:.6f}, {hi:.6f}]")
    else:
        print(f"{name:<34}: {mean_val:.6f}")


# ============================================================
# 18) DETAILS CSV
# ============================================================

def build_eval_details():
    precompute_eval_predictions()
    prepare_answer_semantic_cache()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    questions = _eval_cache["test_questions"]

    qsims = _eval_cache["pred_q_sims"]
    topk_indices = _eval_cache["topk_indices"]
    topk_qsims = _eval_cache["topk_qsims"]
    topk_answers = _eval_cache["topk_answers"]

    ans_cos = _eval_cache["ans_cos"]
    topk_ans_cos = _eval_cache["topk_ans_cos"]

    if RUN_BERTSCORE:
        bert_mean, bert_itemwise = compute_bertscore_f1_at_1()
    else:
        bert_mean, bert_itemwise = None, None

    rows = []

    for i, (q, gold, pred) in enumerate(zip(questions, trues, preds)):
        top_answers = topk_answers[i]

        top_exact_flags = [
            float(norm_for_exact(a) == norm_for_exact(gold))
            for a in top_answers
        ]

        top_sem_flags = [
            float(x >= SEM_THR)
            for x in topk_ans_cos[i].tolist()
        ]

        recall3 = float(any(top_sem_flags))

        mrr3 = 0.0

        for rank, flag in enumerate(top_sem_flags, start=1):
            if flag == 1.0:
                mrr3 = 1.0 / rank
                break

        row = {
            "item_key": stable_item_key(q, gold),
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,

            "QSim": float(qsims[i]),

            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "Top3ExactHit": float(any(top_exact_flags)),
            "TokenF1": float(token_f1(pred, gold)),

            "AnsCos": float(ans_cos[i]),
            "SemHit": float(ans_cos[i] >= SEM_THR),
            "Recall@3": recall3,
            "MRR@3": float(mrr3),

            "Top3_indices": json.dumps(topk_indices[i].tolist(), ensure_ascii=False),
            "Top3_QSims": json.dumps(
                [round(float(x), 6) for x in topk_qsims[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_pred_answers": json.dumps(top_answers, ensure_ascii=False),
            "Top3_AnsCos": json.dumps(
                [round(float(x), 6) for x in topk_ans_cos[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_ExactFlags": json.dumps(top_exact_flags, ensure_ascii=False),
            "Top3_SemFlags": json.dumps(top_sem_flags, ensure_ascii=False),
        }

        if bert_itemwise is not None:
            row["BERTScoreF1"] = float(bert_itemwise[i])

        rows.append(row)

    details_df = pd.DataFrame(rows)

    return details_df


def export_details_csv(path: str):
    details_df = build_eval_details()
    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"CSV сақталды: {path} | rows={len(details_df)}")
    return details_df


# ============================================================
# 19) FINAL EVALUATION
# ============================================================

def evaluate_dataset_qwen_retrieval():
    t0 = time.time()

    precompute_eval_predictions()

    exact, exact_vals = evaluate_exact_at_1()
    top3_exact, top3_exact_vals = evaluate_top3_exact_hit()
    token_f1_mean, token_f1_vals = evaluate_token_f1_at_1()
    qsim_mean, qsim_vals = evaluate_mean_cos_qsim_at_1()

    sem1, sem1_vals = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    recall3, recall3_vals = evaluate_recall_at_3_ans_cos(threshold=SEM_THR)
    mrr3, mrr3_vals = evaluate_mrr_at_3_ans_cos(threshold=SEM_THR)

    if RUN_BERTSCORE:
        bertscore_f1, bertscore_vals = compute_bertscore_f1_at_1()
    else:
        bertscore_f1, bertscore_vals = None, None

    if EXPORT_CSV:
        details_df = export_details_csv(CSV_PATH)
    else:
        details_df = build_eval_details()

    print("\n==================== FULL RESULTS ====================")
    print("Pipeline: Dataset -> Qwen2.5 -> Retrieval")
    print(f"QWEN_MODEL_NAME              : {QWEN_MODEL_NAME}")
    print(f"Dataset rows used            : {len(qa_data)}")
    print(f"Test rows                    : {len(test_answers)}")
    print(f"SEM_THR                      : {SEM_THR}")
    print(f"TOPK_EVAL                    : {TOPK_EVAL}")
    print(f"QWEN_MAX_LENGTH              : {QWEN_MAX_LENGTH}")
    print(f"ENCODE_BATCH_SIZE            : {ENCODE_BATCH_SIZE}")

    fmt_metric("Exact@1", exact, exact_vals)
    fmt_metric("Top-3 ExactHit", top3_exact, top3_exact_vals)
    fmt_metric("TokenF1@1", token_f1_mean, token_f1_vals)
    fmt_metric("MeanCos@1(QSim)", qsim_mean, qsim_vals)

    fmt_metric(f"Semantic@1(ans_cos>={SEM_THR})", sem1, sem1_vals)
    fmt_metric(f"Recall@3(ans_cos>={SEM_THR})", recall3, recall3_vals)
    fmt_metric(f"MRR@3(ans_cos>={SEM_THR})", mrr3, mrr3_vals)

    if bertscore_f1 is not None:
        fmt_metric("BERTScoreF1@1", bertscore_f1, bertscore_vals)
        print(f"BERTScore model              : {_eval_cache.get('bertscore_model_type')}")
    else:
        print("BERTScoreF1@1                : N/A")

    print(f"Evaluation time              : {time.time() - t0:.2f} sec")

    return details_df


# ============================================================
# 20) OPTIONAL INTERACTIVE QA
# ============================================================

RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    print("\nИнтерактив режим қосылды.")
    print("Шығу үшін exit деп жазыңыз.")

    while True:
        user_input = input("\nСұрақ енгізіңіз: ")

        if user_input.strip().lower() == "exit":
            break

        answer, idx, sim, top_idx, top_vals = ask_question_runtime(user_input)

        print("\n================ QA ЖАУАП ================")
        print(answer)

        print("\n---------------- DETAILS ----------------")
        print(f"idx          : {idx}")
        print(f"similarity   : {sim:.4f}")
        print("top_indices  :", top_idx.tolist())
        print("top_scores   :", [round(float(x), 4) for x in top_vals.tolist()])


# ============================================================
# 21) RUN
# ============================================================

details_df = evaluate_dataset_qwen_retrieval()
display(details_df.head(5))

DEVICE: cuda
Mounted at /content/drive
QA JSON файл: grok_python_dataset.json
Барлық QA саны: 5944


,question,answer
0,Python дегеніміз не?,Python - 1991 жылы Гвидо ван Россум жасаған қа...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты -...
2,Python - бағдарламалау тілі қандай салаларда ...,Python - бағдарламалау тілі көптеген салаларда...



1-қадам: Dataset input дайындалуда...
Dataset дайын.


,question,q_input,answer,a_final
0,Python дегеніміз не?,Python дегеніміз не?,Python - 1991 жылы Гвидо ван Россум жасаған қа...,Python-1991 жылы Гвидо ван Россум жасаған қара...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты -...,Python бағдарламалау тілінің негізгі мақсаты-а...
2,Python - бағдарламалау тілі қандай салаларда ...,Python-бағдарламалау тілі қандай салаларда қол...,Python - бағдарламалау тілі көптеген салаларда...,Python-бағдарламалау тілі көптеген салаларда қ...



2-қадам: Qwen2.5 model жүктеледі.
Pipeline: Dataset -> Qwen2.5 -> Retrieval
QWEN_MODEL_NAME: Qwen/Qwen2.5-1.5B-Instruct


...

  0%|          | 0/68 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/38 [00:00<?, ?it/s]

done in 2.82 seconds, 210.62 sentences/sec
BERTScoreF1@1 дайын: 6.89 сек
CSV сақталды: dataset_qwen25_retrieval_full_metrics.csv | rows=595

==================== FULL RESULTS ====================
Pipeline: Dataset -> Qwen2.5 -> Retrieval
QWEN_MODEL_NAME              : Qwen/Qwen2.5-1.5B-Instruct
Dataset rows used            : 5944
Test rows                    : 595
SEM_THR                      : 0.85
TOPK_EVAL                    : 3
QWEN_MAX_LENGTH              : 128
ENCODE_BATCH_SIZE            : 8
Exact@1                           : 0.080672
Top-3 ExactHit                    : 0.089076
TokenF1@1                         : 0.392206
MeanCos@1(QSim)                   : 0.993535
Semantic@1(ans_cos>=0.85)         : 0.784874
Recall@3(ans_cos>=0.85)           : 0.890756
MRR@3(ans_cos>=0.85)              : 0.833053
BERTScoreF1@1                     : 0.799144
BERTScore model              : bert-base-multilingual-cased
Evaluation time              : 38.20 sec


,item_key,test_question,gold_answer,pred_answer,QSim,Exact,Top3ExactHit,TokenF1,AnsCos,SemHit,Recall@3,MRR@3,Top3_indices,Top3_QSims,Top3_pred_answers,Top3_AnsCos,Top3_ExactFlags,Top3_SemFlags,BERTScoreF1
0,d80b7ef941e26fccdbfee4bd547757c0,Сөздіктегі кілт-мағына жұбын қалай жоямыз?,pop() әдісін қолданамыз. Мысалы: sozdik = {'a'...,Кілт арқылы мән тағайындаймыз. Мысалы: sozdik ...,0.996483,0.0,0.0,0.580645,0.960120,1.0,1.0,1.0,"[508, 2273, 3646]","[0.996483, 0.995045, 0.994507]","[""Кілт арқылы мән тағайындаймыз. Мысалы: sozdi...","[0.959577, 0.973294, 0.937104]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.862339
1,ba8c6e07e7c917eb1c47395ecc03d7b4,GitHub Actions-те орналастыруды қалай автоматт...,"AWS, Azure немесе SSH арқылы серверге кодты жі...",env: KEY: value немесе secrets.KEY қолдану.,0.993052,0.0,0.0,0.133333,0.947153,1.0,1.0,1.0,"[4471, 2718, 1341]","[0.993052, 0.991655, 0.99074]","[""env: KEY: value немесе secrets.KEY қолдану.""...","[0.947214, 0.963043, 0.956324]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.671612
2,0ab1409f6d7c527b4d8efb365eca6ff9,Боттың қолжетімділігін қалай мониторинг жасаймыз?,UptimeRobot немесе custom logging арқылы.,Codecov немесе Coveralls арқылы қамту трендтер...,0.989412,0.0,0.0,0.333333,0.949876,1.0,1.0,1.0,"[4702, 1753, 5241]","[0.989412, 0.989412, 0.988804]","[""Codecov немесе Coveralls арқылы қамту трендт...","[0.949776, 0.960969, 0.945796]","[0.0, 0.0, 0.0]","[1.0, 1.0, 1.0]",0.714265
3,8cad88a5baf2b3ddedb2fef91a11ca4c,Функцияда қалай прогресс барын көрсетеміз?,tqdm модулін қолданамыз. Мысалы: from tqdm imp...,def zhup(a): return a% 2 == 0,0.985623,0.0,0.0,0.000000,0.512049,0.0,0.0,0.0,"[4268, 2778, 2736]","[0.985623, 0.984908, 0.98471]","[""def zhup(a): return a% 2 == 0"", ""def on_teks...","[0.512012, 0.575804, 0.571774]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.631771
4,8509cae4e0587b89ab3eeb87fec5b8d0,GET пен POST сұрауларының басты айырмашылығы н...,"GET деректерді алады, POST деректерді жібереді...",Custom headers және деректер құрылымын өзгерту.,0.978838,0.0,0.0,0.125000,0.949346,1.0,1.0,1.0,"[3887, 4805, 1593]","[0.978838, 0.978682, 0.978236]","[""Custom headers және деректер құрылымын өзгер...","[0.949189, 0.812054, 0.955442]","[0.0, 0.0, 0.0]","[1.0, 0.0, 1.0]",0.731320


Dataset -> Qwen2.5 -> Retrieval->Fine tuning

In [ ]:
# @title Dataset -> Qwen2.5 Fine-tuning -> Retrieval | Full Metrics | Google Colab Ready

# ============================================================
# GOOGLE COLAB READY CODE
#
# ҚАТАҢ САҚТАЛҒАН ҚҰРЫЛЫМ:
#
# Dataset
#   -> Qwen2.5 fine-tuning
#   -> Qwen2.5 embedding
#   -> Retrieval
#   -> Evaluation
#
# NO FEMSeg
# NO MAPP
# NO q_dense
#
# Fine-tuning:
#   LoRA / QLoRA
#   Contrastive question-answer alignment
#
# Metrics:
#   Exact@1
#   Top-3 ExactHit
#   TokenF1@1
#   MeanCos@1(QSim)
#   Semantic@1(ans_cos >= 0.85)
#   Recall@3(ans_cos >= 0.85)
#   MRR@3(ans_cos >= 0.85)
#   BERTScoreF1@1
# ============================================================


# ============================================================
# 0) INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers accelerate bitsandbytes peft bert-score scikit-learn


# ============================================================
# 1) IMPORTS AND ENVIRONMENT
# ============================================================

import os
os.environ["WANDB_DISABLED"] = "true"

import re
import glob
import time
import json
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

from bert_score import score as bert_score

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 2) SPEED / SEED SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE != "cuda":
    raise RuntimeError(
        "Qwen2.5 fine-tuning үшін GPU керек. Colab-та Runtime -> Change runtime type -> GPU таңдаңыз."
    )


# ============================================================
# 3) CONFIG
# ============================================================

DATA_JSON_PATH = "grok_python_dataset.json"

TEST_SIZE = 0.10
SEM_THR = 0.85
TOPK_EVAL = 3

# 5000 QA үшін лайықты параметрлер
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

USE_4BIT = True

NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
TEMPERATURE = 0.05
MAX_GRAD_NORM = 1.0

MAX_Q_LEN = 128
MAX_A_LEN = 192

EVAL_ENCODE_BATCH_SIZE = 8
SIM_BATCH_SIZE = 256

QWEN_MAX_LENGTH_FOR_RETRIEVAL = 160

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Evaluation
RUN_BERTSCORE = True
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
# Сапалырақ, бірақ ауыр нұсқа:
# BERTSCORE_MODEL_TYPE = "xlm-roberta-large"

BERTSCORE_BATCH_SIZE = 16

RUN_BOOTSTRAP_CI = False
BOOT_N = 500

EXPORT_CSV = True
CSV_PATH = "qwen25_finetuned_retrieval_full_metrics.csv"

SAVE_LORA = True
LORA_SAVE_DIR = "/content/drive/MyDrive/KAZ_MORPH/qwen25_15b_lora_qa_retrieval" if IN_COLAB else "./qwen25_15b_lora_qa_retrieval"

USE_EMBEDDING_CACHE = False


# ============================================================
# 4) GOOGLE DRIVE
# ============================================================

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)

CACHE_DIR = Path("/content/drive/MyDrive/KAZ_MORPH/qwen_ft_cache") if IN_COLAB else Path("./qwen_ft_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 5) ROBUST JSON LOADING
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p

    candidates = [
        f"/content/{p}",
        f"/content/drive/MyDrive/{p}",
        f"/content/drive/MyDrive/KAZ_MORPH/{p}",
    ]

    for c in candidates:
        if Path(c).exists():
            return c

    name = Path(p).name
    hits = glob.glob(f"**/{name}", recursive=True)

    if hits:
        return hits[0]

    near = glob.glob("**/*.json", recursive=True)

    raise FileNotFoundError(
        f"Файл табылмады: {p}\n"
        f"Табылған .json файлдар:\n" + "\n".join(near[:30])
    )


def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)


def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)


def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text


def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None

    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()

        if q and a:
            return {"question": q, "answer": a}

    return None


def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)

    if normalized is not None:
        return [normalized]

    if isinstance(obj, list):
        out = []

        for item in obj:
            norm = _normalize_record(item)

            if norm is not None:
                out.append(norm)

        return out

    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                out = []

                for item in obj[key]:
                    norm = _normalize_record(item)

                    if norm is not None:
                        out.append(norm)

                return out

    return []


def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    obj = json.loads(_normalize_json_text(text))
    records = _extract_records(obj)

    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")

    return records


def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx = 0
    records = []

    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1

        if idx >= len(text):
            break

        obj, end = decoder.raw_decode(text, idx)
        idx = end

        extracted = _extract_records(obj)

        if extracted:
            records.extend(extracted)

    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")

    return records


def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records = []
    bad_lines = []

    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()

        if not s:
            continue

        if s.endswith(","):
            s = s[:-1].strip()

        s = _normalize_json_text(s)

        try:
            obj = json.loads(s)
            extracted = _extract_records(obj)

            if extracted:
                records.extend(extracted)
            else:
                bad_lines.append((line_no, "QA өрістері табылмады", s[:200]))

        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))

    if not records:
        preview = "\n".join(
            [f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]]
        )

        raise ValueError(
            "Файлдан бірде-бір QA жазба оқылмады.\n"
            f"Алғашқы қате жолдар:\n{preview}"
        )

    if bad_lines:
        print(f"Ескерту: {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")

    return records


def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []

    for parser in (
        _parse_as_single_json,
        _parse_as_multiple_json_objects,
        _parse_line_by_line,
    ):
        try:
            records = parser(text)

            if records:
                return records

        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")

    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


data_path = find_data_path(DATA_JSON_PATH)
print("QA JSON файл:", data_path)

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON ішінде question/answer немесе instruction/response болуы керек."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"] = qa_data["answer"].astype(str).str.strip()

qa_data = qa_data[
    (qa_data["question"] != "") &
    (qa_data["answer"] != "")
].reset_index(drop=True)

print("Барлық QA саны:", len(qa_data))
display(qa_data.head(3))


# ============================================================
# 6) BASIC TEXT PROCESSING AND METRICS
# ============================================================

_punct_space_left = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space = re.compile(r"\s+")


def clean_text(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_text(text).lower()).strip()


def tokens(text: str) -> List[str]:
    t = clean_text(text).lower()

    return re.findall(
        r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+",
        t
    )


def token_f1(pred: str, gold: str) -> float:
    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    pc = Counter(p)
    gc = Counter(g)

    inter = sum((pc & gc).values())

    if inter == 0:
        return 0.0

    precision = inter / max(1, len(p))
    recall = inter / max(1, len(g))

    return (2 * precision * recall) / (precision + recall + 1e-12)


def stable_item_key(test_question: str, gold_answer: str) -> str:
    raw = norm_for_exact(test_question) + " ||| " + norm_for_exact(gold_answer)
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


# ============================================================
# 7) DATASET PREPARATION
# ============================================================
# ҚҰРЫЛЫМ:
# Dataset -> Qwen2.5 fine-tuning -> Retrieval
#
# Мұнда FEMSeg жоқ.
# Мұнда MAPP жоқ.
# Мұнда q_dense жоқ.

print("\n1-қадам: Dataset input дайындалуда...")

qa_data["q_input"] = qa_data["question"].map(clean_text)
qa_data["a_final"] = qa_data["answer"].map(clean_text)

print("Dataset дайын.")
display(qa_data[["question", "q_input", "answer", "a_final"]].head(3))


# ============================================================
# 8) TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    qa_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\nTRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED}")


# ============================================================
# 9) LOAD Qwen2.5 MODEL
# ============================================================

print("\n2-қадам: Qwen2.5 model жүктеледі.")
print("Pipeline: Dataset -> Qwen2.5 fine-tuning -> Retrieval")
print("QWEN_MODEL_NAME:", QWEN_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

else:
    qwen_model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(DEVICE)

qwen_model.config.use_cache = False

if USE_4BIT:
    qwen_model = prepare_model_for_kbit_training(
        qwen_model,
        use_gradient_checkpointing=True
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)

qwen_model = get_peft_model(qwen_model, lora_config)

try:
    qwen_model.print_trainable_parameters()
except Exception:
    pass

qwen_model.train()

QWEN_DEVICE = next(qwen_model.parameters()).device
print("Qwen2.5 + LoRA дайын.")
print("QWEN_DEVICE:", QWEN_DEVICE)


# ============================================================
# 10) EMBEDDING FUNCTIONS
# ============================================================

def last_token_pool(
    last_hidden_state: torch.Tensor,
    attention_mask: torch.Tensor
) -> torch.Tensor:
    lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(
        last_hidden_state.size(0),
        device=last_hidden_state.device
    )
    pooled = last_hidden_state[batch_idx, lengths]
    return pooled


def encode_from_batch(batch_enc: Dict[str, torch.Tensor]) -> torch.Tensor:
    out = qwen_model(
        **batch_enc,
        output_hidden_states=False,
        return_dict=True
    )

    pooled = last_token_pool(
        out.last_hidden_state,
        batch_enc["attention_mask"]
    )

    pooled = pooled.float()
    pooled = F.normalize(pooled, p=2, dim=1)

    return pooled


@torch.inference_mode()
def qwen_encode(
    texts: List[str],
    batch_size: int = EVAL_ENCODE_BATCH_SIZE,
    normalize: bool = True,
    max_length: int = QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress: bool = True
) -> np.ndarray:
    qwen_model.eval()

    all_vecs = []
    n = len(texts)
    t0 = time.time()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_texts = [str(x) for x in texts[start:end]]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {
            k: v.to(QWEN_DEVICE)
            for k, v in enc.items()
        }

        out = qwen_model(
            **enc,
            output_hidden_states=False,
            return_dict=True
        )

        pooled = last_token_pool(
            out.last_hidden_state,
            enc["attention_mask"]
        ).float()

        if normalize:
            pooled = F.normalize(pooled, p=2, dim=1)

        all_vecs.append(
            pooled.detach().cpu().numpy()
        )

        if show_progress and ((end % 1000 == 0) or end == n):
            print(f"Encoded {end}/{n} | {time.time() - t0:.1f} сек")

    return np.vstack(all_vecs).astype(np.float32)


# ============================================================
# 11) TRAINING DATASET
# ============================================================

class QAPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.questions = df["q_input"].tolist()
        self.answers = df["a_final"].tolist()

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return {
            "question": self.questions[idx],
            "answer": self.answers[idx],
        }


def collate_qa_batch(batch):
    questions = [
        item["question"]
        for item in batch
    ]

    answers = [
        item["answer"]
        for item in batch
    ]

    q_enc = tokenizer(
        questions,
        padding=True,
        truncation=True,
        max_length=MAX_Q_LEN,
        return_tensors="pt"
    )

    a_enc = tokenizer(
        answers,
        padding=True,
        truncation=True,
        max_length=MAX_A_LEN,
        return_tensors="pt"
    )

    q_enc = {
        k: v.to(QWEN_DEVICE)
        for k, v in q_enc.items()
    }

    a_enc = {
        k: v.to(QWEN_DEVICE)
        for k, v in a_enc.items()
    }

    return q_enc, a_enc


train_dataset = QAPairDataset(train_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_qa_batch,
    drop_last=True,
)


# ============================================================
# 12) CONTRASTIVE FINE-TUNING
# ============================================================

trainable_params = [
    p for p in qwen_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_update_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
warmup_steps = int(total_update_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps
)

print("\n==================== TRAINING CONFIG ====================")
print(f"Train size              : {len(train_df)}")
print(f"Test size               : {len(test_df)}")
print(f"Epochs                  : {NUM_EPOCHS}")
print(f"Train batch size         : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation    : {GRAD_ACCUM_STEPS}")
print(f"Effective batch size     : {EFFECTIVE_BATCH_SIZE}")
print(f"Learning rate            : {LEARNING_RATE}")
print(f"Warmup steps             : {warmup_steps}")
print(f"Total update steps       : {total_update_steps}")
print(f"Temperature              : {TEMPERATURE}")
print(f"LoRA r                   : {LORA_R}")
print(f"LoRA alpha               : {LORA_ALPHA}")
print(f"LoRA dropout             : {LORA_DROPOUT}")
print("==========================================================")


def contrastive_loss(q_emb: torch.Tensor, a_emb: torch.Tensor, temperature: float = TEMPERATURE):
    logits = torch.matmul(q_emb, a_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_q_to_a = F.cross_entropy(logits, labels)
    loss_a_to_q = F.cross_entropy(logits.T, labels)

    loss = 0.5 * (loss_q_to_a + loss_a_to_q)

    return loss


qwen_model.train()

global_step = 0
running_loss = 0.0

t_train = time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS} басталды...")
    epoch_t0 = time.time()

    for step, (q_enc, a_enc) in enumerate(train_loader, start=1):
        with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
            q_emb = encode_from_batch(q_enc)
            a_emb = encode_from_batch(a_enc)

            loss = contrastive_loss(q_emb, a_emb)
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if global_step % 25 == 0:
                avg_loss = running_loss / 25
                running_loss = 0.0
                lr_now = scheduler.get_last_lr()[0]
                print(
                    f"epoch={epoch} | update_step={global_step}/{total_update_steps} | "
                    f"loss={avg_loss:.4f} | lr={lr_now:.2e}"
                )

    print(f"Epoch {epoch} аяқталды | уақыт: {time.time() - epoch_t0:.1f} сек")

print(f"\nFine-tuning аяқталды. Жалпы уақыт: {time.time() - t_train:.1f} сек")

if SAVE_LORA:
    Path(LORA_SAVE_DIR).mkdir(parents=True, exist_ok=True)
    qwen_model.save_pretrained(LORA_SAVE_DIR)
    tokenizer.save_pretrained(LORA_SAVE_DIR)
    print(f"LoRA adapter сақталды: {LORA_SAVE_DIR}")

torch.cuda.empty_cache()


# ============================================================
# 13) PRECOMPUTE EMBEDDINGS AFTER FINE-TUNING
# ============================================================

qwen_model.eval()

train_questions = train_df["q_input"].tolist()
test_questions = test_df["q_input"].tolist()

train_answers = train_df["a_final"].tolist()
test_answers = test_df["a_final"].tolist()

print("\n3-қадам: Fine-tuned Qwen2.5 арқылы question embeddings есептелуде...")

t0 = time.time()

train_q_emb = qwen_encode(
    train_questions,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

test_q_emb = qwen_encode(
    test_questions,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

print(f"Question embeddings дайын: {time.time() - t0:.2f} сек")


print("\n4-қадам: Fine-tuned Qwen2.5 арқылы answer embeddings есептелуде...")

t0 = time.time()

train_a_emb = qwen_encode(
    train_answers,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

test_a_emb = qwen_encode(
    test_answers,
    batch_size=EVAL_ENCODE_BATCH_SIZE,
    normalize=True,
    max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
    show_progress=True
)

print(f"Answer embeddings дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 14) RETRIEVAL
# ============================================================

def get_topk_sorted(
    scores: np.ndarray,
    k: int
) -> Tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[0])

    if k <= 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx = np.argpartition(-scores, kth=k - 1)[:k]
    vals = scores[idx]
    order = np.argsort(-vals)

    return idx[order].astype(np.int32), vals[order].astype(np.float32)


_eval_cache = {}


def precompute_eval_predictions():
    if "pred_answers" in _eval_cache:
        return

    print("\n5-қадам: TEST -> TRAIN retrieval есептелуде...")
    t0 = time.time()

    pred_indices = []
    pred_q_sims = []
    pred_answers = []

    topk_indices_list = []
    topk_qsims_list = []
    topk_answers_list = []

    n_test = len(test_q_emb)
    k_eval = min(TOPK_EVAL, train_q_emb.shape[0])

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]

        sims = batch @ train_q_emb.T

        local_top_idx = np.argpartition(
            -sims,
            kth=k_eval - 1,
            axis=1
        )[:, :k_eval]

        row_ids = np.arange(end - start)[:, None]

        local_top_vals = sims[
            row_ids,
            local_top_idx
        ]

        order = np.argsort(-local_top_vals, axis=1)

        local_top_idx = np.take_along_axis(
            local_top_idx,
            order,
            axis=1
        )

        local_top_vals = np.take_along_axis(
            local_top_vals,
            order,
            axis=1
        )

        for r in range(end - start):
            row_indices = local_top_idx[r].astype(int).tolist()
            row_sims = local_top_vals[r].astype(float).tolist()
            row_answers = [
                train_answers[i]
                for i in row_indices
            ]

            pred_indices.append(row_indices[0])
            pred_q_sims.append(row_sims[0])
            pred_answers.append(row_answers[0])

            topk_indices_list.append(row_indices)
            topk_qsims_list.append(row_sims)
            topk_answers_list.append(row_answers)

    _eval_cache["pred_indices"] = np.array(pred_indices, dtype=np.int32)
    _eval_cache["pred_q_sims"] = np.array(pred_q_sims, dtype=np.float32)
    _eval_cache["pred_answers"] = pred_answers
    _eval_cache["true_answers"] = test_answers
    _eval_cache["test_questions"] = [
        clean_text(q)
        for q in test_questions
    ]
    _eval_cache["topk_indices"] = np.array(topk_indices_list, dtype=np.int32)
    _eval_cache["topk_qsims"] = np.array(topk_qsims_list, dtype=np.float32)
    _eval_cache["topk_answers"] = topk_answers_list

    print(f"Retrieval дайын: {time.time() - t0:.2f} сек")


# ============================================================
# 15) ANSWER SEMANTIC METRICS
# ============================================================

def prepare_answer_semantic_cache():
    if "ans_cos" in _eval_cache and "topk_ans_cos" in _eval_cache:
        return

    precompute_eval_predictions()

    pred_indices = _eval_cache["pred_indices"]
    topk_indices = _eval_cache["topk_indices"]

    pred_a_emb = train_a_emb[pred_indices]
    true_a_emb = test_a_emb

    ans_cos = np.sum(
        pred_a_emb * true_a_emb,
        axis=1
    )

    topk_a_emb = train_a_emb[topk_indices]

    topk_ans_cos = np.sum(
        topk_a_emb * true_a_emb[:, None, :],
        axis=2
    )

    _eval_cache["ans_cos"] = ans_cos.astype(np.float32)
    _eval_cache["topk_ans_cos"] = topk_ans_cos.astype(np.float32)


# ============================================================
# 16) BERTScore
# ============================================================

def compute_bertscore_f1_at_1():
    if "bertscore_itemwise" in _eval_cache and _eval_cache["bertscore_itemwise"] is not None:
        itemwise = _eval_cache["bertscore_itemwise"]
        return float(np.mean(itemwise)), itemwise

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n6-қадам: BERTScoreF1@1 есептелуде...")
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)

    t0 = time.time()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        _, _, f1 = bert_score(
            preds,
            trues,
            model_type=BERTSCORE_MODEL_TYPE,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    except Exception as e:
        print("Бірінші BERTScore есептеу сәтсіз болды:", repr(e))
        print("Fallback: lang='kk' арқылы есептеу басталды...")

        _, _, f1 = bert_score(
            preds,
            trues,
            lang="kk",
            batch_size=BERTSCORE_BATCH_SIZE,
            rescale_with_baseline=False,
            device=device,
            verbose=True
        )

    if hasattr(f1, "detach"):
        itemwise = f1.detach().cpu().numpy().astype(float)
    else:
        itemwise = np.array(f1).astype(float)

    _eval_cache["bertscore_itemwise"] = itemwise
    _eval_cache["bertscore_f1_mean"] = float(np.mean(itemwise))
    _eval_cache["bertscore_model_type"] = BERTSCORE_MODEL_TYPE

    print(f"BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")

    return float(np.mean(itemwise)), itemwise


# ============================================================
# 17) METRICS
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        float(norm_for_exact(p) == norm_for_exact(t))
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_top3_exact_hit():
    precompute_eval_predictions()

    topk_answers = _eval_cache["topk_answers"]
    trues = _eval_cache["true_answers"]

    hits = []

    for row_answers, gold in zip(topk_answers, trues):
        hit = any(
            norm_for_exact(a) == norm_for_exact(gold)
            for a in row_answers
        )

        hits.append(float(hit))

    return float(np.mean(hits)), hits


def evaluate_token_f1_at_1():
    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    scores = [
        token_f1(p, t)
        for p, t in zip(preds, trues)
    ]

    return float(np.mean(scores)), scores


def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()

    vals = _eval_cache["pred_q_sims"].astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_semantic_at_1_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    ans_cos = _eval_cache["ans_cos"]
    vals = (ans_cos >= threshold).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_recall_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    vals = np.any(topk_ans_cos >= threshold, axis=1).astype(float).tolist()

    return float(np.mean(vals)), vals


def evaluate_mrr_at_3_ans_cos(threshold: float = SEM_THR):
    prepare_answer_semantic_cache()

    topk_ans_cos = _eval_cache["topk_ans_cos"]
    flags = (topk_ans_cos >= threshold).astype(np.int32)

    rr = []

    for row in flags:
        val = 0.0

        for rank, flag in enumerate(row.tolist(), start=1):
            if flag == 1:
                val = 1.0 / rank
                break

        rr.append(val)

    return float(np.mean(rr)), rr


# ============================================================
# 18) BOOTSTRAP CI
# ============================================================

def bootstrap_ci_mean(
    values: List[float],
    n_boot: int = BOOT_N,
    seed: int = SEED
) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    n = len(arr)

    if n == 0:
        return float("nan"), float("nan")

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = float(np.mean(arr[idx]))

    lo = float(np.percentile(boots, 2.5))
    hi = float(np.percentile(boots, 97.5))

    return lo, hi


def fmt_metric(name: str, mean_val, vals=None):
    if RUN_BOOTSTRAP_CI and vals is not None:
        lo, hi = bootstrap_ci_mean(vals)
        print(f"{name:<34}: {mean_val:.6f} | 95% CI [{lo:.6f}, {hi:.6f}]")
    else:
        print(f"{name:<34}: {mean_val:.6f}")


# ============================================================
# 19) DETAILS CSV
# ============================================================

def build_eval_details():
    precompute_eval_predictions()
    prepare_answer_semantic_cache()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    questions = _eval_cache["test_questions"]

    qsims = _eval_cache["pred_q_sims"]
    topk_indices = _eval_cache["topk_indices"]
    topk_qsims = _eval_cache["topk_qsims"]
    topk_answers = _eval_cache["topk_answers"]

    ans_cos = _eval_cache["ans_cos"]
    topk_ans_cos = _eval_cache["topk_ans_cos"]

    if RUN_BERTSCORE:
        bert_mean, bert_itemwise = compute_bertscore_f1_at_1()
    else:
        bert_mean, bert_itemwise = None, None

    rows = []

    for i, (q, gold, pred) in enumerate(zip(questions, trues, preds)):
        top_answers = topk_answers[i]

        top_exact_flags = [
            float(norm_for_exact(a) == norm_for_exact(gold))
            for a in top_answers
        ]

        top_sem_flags = [
            float(x >= SEM_THR)
            for x in topk_ans_cos[i].tolist()
        ]

        recall3 = float(any(top_sem_flags))

        mrr3 = 0.0

        for rank, flag in enumerate(top_sem_flags, start=1):
            if flag == 1.0:
                mrr3 = 1.0 / rank
                break

        row = {
            "item_key": stable_item_key(q, gold),
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,

            "QSim": float(qsims[i]),

            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "Top3ExactHit": float(any(top_exact_flags)),
            "TokenF1": float(token_f1(pred, gold)),

            "AnsCos": float(ans_cos[i]),
            "SemHit": float(ans_cos[i] >= SEM_THR),
            "Recall@3": recall3,
            "MRR@3": float(mrr3),

            "Top3_indices": json.dumps(topk_indices[i].tolist(), ensure_ascii=False),
            "Top3_QSims": json.dumps(
                [round(float(x), 6) for x in topk_qsims[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_pred_answers": json.dumps(top_answers, ensure_ascii=False),
            "Top3_AnsCos": json.dumps(
                [round(float(x), 6) for x in topk_ans_cos[i].tolist()],
                ensure_ascii=False
            ),
            "Top3_ExactFlags": json.dumps(top_exact_flags, ensure_ascii=False),
            "Top3_SemFlags": json.dumps(top_sem_flags, ensure_ascii=False),
        }

        if bert_itemwise is not None:
            row["BERTScoreF1"] = float(bert_itemwise[i])

        rows.append(row)

    details_df = pd.DataFrame(rows)

    return details_df


def export_details_csv(path: str):
    details_df = build_eval_details()
    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"CSV сақталды: {path} | rows={len(details_df)}")
    return details_df


# ============================================================
# 20) FINAL EVALUATION
# ============================================================

def evaluate_finetuned_qwen_retrieval():
    t0 = time.time()

    precompute_eval_predictions()

    exact, exact_vals = evaluate_exact_at_1()
    top3_exact, top3_exact_vals = evaluate_top3_exact_hit()
    token_f1_mean, token_f1_vals = evaluate_token_f1_at_1()
    qsim_mean, qsim_vals = evaluate_mean_cos_qsim_at_1()

    sem1, sem1_vals = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    recall3, recall3_vals = evaluate_recall_at_3_ans_cos(threshold=SEM_THR)
    mrr3, mrr3_vals = evaluate_mrr_at_3_ans_cos(threshold=SEM_THR)

    if RUN_BERTSCORE:
        bertscore_f1, bertscore_vals = compute_bertscore_f1_at_1()
    else:
        bertscore_f1, bertscore_vals = None, None

    if EXPORT_CSV:
        details_df = export_details_csv(CSV_PATH)
    else:
        details_df = build_eval_details()

    print("\n==================== FINE-TUNED FULL RESULTS ====================")
    print("Pipeline: Dataset -> Qwen2.5 fine-tuning -> Retrieval")
    print(f"QWEN_MODEL_NAME              : {QWEN_MODEL_NAME}")
    print(f"Dataset rows used            : {len(qa_data)}")
    print(f"Train rows                   : {len(train_df)}")
    print(f"Test rows                    : {len(test_answers)}")
    print(f"Epochs                       : {NUM_EPOCHS}")
    print(f"Effective batch size          : {EFFECTIVE_BATCH_SIZE}")
    print(f"Learning rate                 : {LEARNING_RATE}")
    print(f"SEM_THR                       : {SEM_THR}")
    print(f"TOPK_EVAL                     : {TOPK_EVAL}")

    fmt_metric("Exact@1", exact, exact_vals)
    fmt_metric("Top-3 ExactHit", top3_exact, top3_exact_vals)
    fmt_metric("TokenF1@1", token_f1_mean, token_f1_vals)
    fmt_metric("MeanCos@1(QSim)", qsim_mean, qsim_vals)

    fmt_metric(f"Semantic@1(ans_cos>={SEM_THR})", sem1, sem1_vals)
    fmt_metric(f"Recall@3(ans_cos>={SEM_THR})", recall3, recall3_vals)
    fmt_metric(f"MRR@3(ans_cos>={SEM_THR})", mrr3, mrr3_vals)

    if bertscore_f1 is not None:
        fmt_metric("BERTScoreF1@1", bertscore_f1, bertscore_vals)
        print(f"BERTScore model              : {_eval_cache.get('bertscore_model_type')}")
    else:
        print("BERTScoreF1@1                : N/A")

    print(f"Evaluation time              : {time.time() - t0:.2f} sec")
    print("===============================================================")

    return details_df


# ============================================================
# 21) OPTIONAL INTERACTIVE QA
# ============================================================

_query_emb_cache = {}


def get_query_embedding(question: str) -> np.ndarray:
    q = clean_text(question)

    if q in _query_emb_cache:
        return _query_emb_cache[q]

    qv = qwen_encode(
        [q],
        batch_size=1,
        normalize=True,
        max_length=QWEN_MAX_LENGTH_FOR_RETRIEVAL,
        show_progress=False
    )

    _query_emb_cache[q] = qv

    return qv


def ask_question_runtime(question: str):
    qv = get_query_embedding(question)
    sims = (train_q_emb @ qv.T).squeeze(1)

    top_idx, top_vals = get_topk_sorted(
        sims,
        TOPK_EVAL
    )

    best_idx = int(top_idx[0])
    best_sim = float(top_vals[0])
    best_answer = train_answers[best_idx]

    return best_answer, best_idx, best_sim, top_idx, top_vals


RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    print("\nИнтерактив режим қосылды.")
    print("Шығу үшін exit деп жазыңыз.")

    while True:
        user_input = input("\nСұрақ енгізіңіз: ")

        if user_input.strip().lower() == "exit":
            break

        answer, idx, sim, top_idx, top_vals = ask_question_runtime(user_input)

        print("\n================ QA ЖАУАП ================")
        print(answer)

        print("\n---------------- DETAILS ----------------")
        print(f"idx          : {idx}")
        print(f"similarity   : {sim:.4f}")
        print("top_indices  :", top_idx.tolist())
        print("top_scores   :", [round(float(x), 4) for x in top_vals.tolist()])


# ============================================================
# 22) RUN FINAL EVALUATION
# ============================================================

details_df = evaluate_finetuned_qwen_retrieval()
display(details_df.head(5))

DEVICE: cuda
Mounted at /content/drive
QA JSON файл: grok_python_dataset.json
Барлық QA саны: 5944


,question,answer
0,Python дегеніміз не?,Python - 1991 жылы Гвидо ван Россум жасаған қа...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты -...
2,Python - бағдарламалау тілі қандай салаларда ...,Python - бағдарламалау тілі көптеген салаларда...



1-қадам: Dataset input дайындалуда...
Dataset дайын.


,question,q_input,answer,a_final
0,Python дегеніміз не?,Python дегеніміз не?,Python - 1991 жылы Гвидо ван Россум жасаған қа...,Python-1991 жылы Гвидо ван Россум жасаған қара...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты -...,Python бағдарламалау тілінің негізгі мақсаты-а...
2,Python - бағдарламалау тілі қандай салаларда ...,Python-бағдарламалау тілі қандай салаларда қол...,Python - бағдарламалау тілі көптеген салаларда...,Python-бағдарламалау тілі көптеген салаларда қ...



TRAIN: 5349 | TEST: 595 | seed=42

2-қадам: Qwen2.5 model жүктеледі.
Pipeline: Dataset -> Qwen2.5 fine-tuning -> Retrieval
QWEN_MODEL_NAME: Qwen/Qwen2.5-1.5B-Instruct


...

  0%|          | 0/68 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/38 [00:00<?, ?it/s]

done in 2.65 seconds, 224.23 sentences/sec
BERTScoreF1@1 дайын: 6.78 сек
CSV сақталды: qwen25_finetuned_retrieval_full_metrics.csv | rows=595

==================== FINE-TUNED FULL RESULTS ====================
Pipeline: Dataset -> Qwen2.5 fine-tuning -> Retrieval
QWEN_MODEL_NAME              : Qwen/Qwen2.5-1.5B-Instruct
Dataset rows used            : 5944
Train rows                   : 5349
Test rows                    : 595
Epochs                       : 2
Effective batch size          : 16
Learning rate                 : 0.0002
SEM_THR                       : 0.85
TOPK_EVAL                     : 3
Exact@1                           : 0.082353
Top-3 ExactHit                    : 0.100840
TokenF1@1                         : 0.449984
MeanCos@1(QSim)                   : 0.887136
Semantic@1(ans_cos>=0.85)         : 0.331092
Recall@3(ans_cos>=0.85)           : 0.435294
MRR@3(ans_cos>=0.85)              : 0.377031
BERTScoreF1@1                     : 0.815771
BERTScore model              : ber

,item_key,test_question,gold_answer,pred_answer,QSim,Exact,Top3ExactHit,TokenF1,AnsCos,SemHit,Recall@3,MRR@3,Top3_indices,Top3_QSims,Top3_pred_answers,Top3_AnsCos,Top3_ExactFlags,Top3_SemFlags,BERTScoreF1
0,d80b7ef941e26fccdbfee4bd547757c0,Сөздіктегі кілт-мағына жұбын қалай жоямыз?,pop() әдісін қолданамыз. Мысалы: sozdik = {'a'...,Кілт арқылы мән тағайындаймыз. Мысалы: sozdik ...,0.889711,0.0,0.0,0.580645,0.743903,0.0,0.0,0.0,"[508, 1964, 2273]","[0.889711, 0.86372, 0.854659]","[""Кілт арқылы мән тағайындаймыз. Мысалы: sozdi...","[0.743903, 0.665015, 0.622967]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.862339
1,ba8c6e07e7c917eb1c47395ecc03d7b4,GitHub Actions-те орналастыруды қалай автоматт...,"AWS, Azure немесе SSH арқылы серверге кодты жі...",.github/workflows/ci.yml файлына build және te...,0.882354,0.0,0.0,0.000000,0.488386,0.0,0.0,0.0,"[337, 2565, 5050]","[0.882354, 0.864302, 0.858107]","["".github/workflows/ci.yml файлына build және ...","[0.488386, 0.668687, 0.652155]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.708983
2,0ab1409f6d7c527b4d8efb365eca6ff9,Боттың қолжетімділігін қалай мониторинг жасаймыз?,UptimeRobot немесе custom logging арқылы.,GitHub Actions арқылы Markdown синтаксисін тек...,0.773897,0.0,0.0,0.181818,0.228176,0.0,0.0,0.0,"[4984, 2064, 5059]","[0.773897, 0.751356, 0.727466]","[""GitHub Actions арқылы Markdown синтаксисін т...","[0.228176, 0.617274, 0.602672]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.718942
3,8cad88a5baf2b3ddedb2fef91a11ca4c,Функцияда қалай прогресс барын көрсетеміз?,tqdm модулін қолданамыз. Мысалы: from tqdm imp...,logging модулін қолданамыз. Мысалы: import log...,0.467413,0.0,0.0,0.470588,0.319949,0.0,0.0,0.0,"[4852, 5019, 3918]","[0.467413, 0.439972, 0.412752]","[""logging модулін қолданамыз. Мысалы: import l...","[0.319949, 0.028836, 0.366]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.810501
4,8509cae4e0587b89ab3eeb87fec5b8d0,GET пен POST сұрауларының басты айырмашылығы н...,"GET деректерді алады, POST деректерді жібереді...",Custom headers және деректер құрылымын өзгерту.,0.739669,0.0,0.0,0.125000,0.696971,0.0,0.0,0.0,"[3887, 1339, 1414]","[0.739669, 0.713668, 0.709514]","[""Custom headers және деректер құрылымын өзгер...","[0.696971, 0.828392, 0.631923]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",0.731320
